# 76 · AKI along the PT on the package pseudospace (Figure 6)

**The question.** Along the package PT axis, where do ischaemia–reperfusion (IRI) mice lose PT units, lose transport
gene expression and raise a cell-cycle program, relative to control mice? Do the two responses sit at different
positions ("decoupling")? The user's outline (Results 5) claims "progressive, segment-dependent decoupling of
transport functions and cell-cycle activation"; Figure 6 of `docs/paper/figure_plan_methods.md` was blocked on an AKI
coordinate, which notebook 70 now provides.

**The frozen protocol.** `docs/results/pt-package-aki.md` (commit 21dbc94), fixed before any fit, including the
coordinator's decisions O1–O6. Every claim rule (section 10 there) is applied as written; nothing is tuned.

**What changes from notebook 71's AKI arm.** Only the gene universe: notebook 71 restricted AKI to the human vs mouse
universe, which excludes Havcr1, Lcn2, Mki67 and Top2a. Here the universe is the mouse ortholog panel with AKI's own
detection rule (≥ 2% equal-specimen detection), mt genes removed. The gene model, the positional F-test, cameraPR,
the relabelings and the pass rule are notebook 71's, unchanged. **Notebook 71's AKI calls stay the record for the
method comparison; this notebook's calls serve Figure 6 only.**

**Sections.**
1. Setup, inputs, guards and the mouse-panel universe.
2. The gene model and the positional F-test (notebook 71's sections 2–3) on the primary cohort; the reuse guard.
3. The screen's pass rule and the comparison with notebook 71.
4. The programs: T, T_PT, C, R.
5. Segment units and program profiles.
6. (b) The positive control.
7. Placement of the units without a PT position (c19, c15).
8. (c) PT unit density along the axis.
9. (d) Transport loss, the cell-cycle program and decoupling; the two cell-cycle readings.
10. Cohort facts.
11. Sensitivities: c19 added (two arms) and the package DPT.
12. Claims A1–A7.
13. Figure 6.
14. How the protocol was read.
15. Summary.
16. Run record.

**Assumptions.** Two control and two IRI male mice; the user states that the two IR mice were injured at the same
severity. Specimens are the replicates and every result is descriptive. The coordinate is treated as exact. The
package labels an injured-PT cluster (c19) cTAL and drops a mixed injured/stromal cluster (c15), so the most injured
units are outside the PT cohort: section 7 and section 11 quantify this. The IRI time point is unknown, so the
cell-cycle program is named neutrally.

## 1 · Setup

In [ ]:
import argparse
import json
import os
import re
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
package_aki = results_root / 'package_pseudospace' / 'aki'
run_aki = results_root / 'package_pseudospace' / 'runs' / 'aki'
upstream13 = results_root / 'minimal_pt_scfates'
T71 = results_root / 'pt_pathways_package_pseudospace'                # notebook 71: read only (tables, stage cache)
T72 = results_root / 'pt_package_coordinate_validation' / 'tables'    # notebook 72: quoted only
HGNC = {752: data_root / 'external' / 'hgnc' / 'hgnc_group_752.tsv', 417: data_root / 'external' / 'hgnc' / 'hgnc_group_417.tsv'}
HGNC_SHA = {752: '131e82911a0840e98f70c6c3075059d0a69a8709c0a3c43db69029aed3a97bee',
            417: 'cc6cdac305cd75465de8ae34faac429a0356db916171bfeba4ec8d4ce82cf28a'}     # notebook 74
output = results_root / 'pt_package_aki'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)
CONTROLS, IRS = ['Ctrl1A2', 'Ctrl1A4'], ['IR2A2', 'IR2A4']
SPECIMENS = CONTROLS + IRS
EXPORT = package_aki / 'pt_units.h5ad'
LIBRARIES = ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')
REQUIRED = [EXPORT, package_aki / 'all_units_positions.csv', package_aki / 'tables' / 'pt_stretch.csv',
            run_aki / '03_integration' / 'pass1_labelled.h5ad', run_aki / '02_qc' / 'units_qc.h5ad',
            *[run_aki / '01_units' / f'{s}_units.h5ad' for s in SPECIMENS],
            upstream13 / 'ortholog_map_used.csv', T72 / 'criteria_aki.csv',
            *[T71 / 'tables' / name for name in ('universe_aki_vs_control.csv', 'gene_sets_aki_vs_control.json',
                                                 'gene_statistics_aki_vs_control.csv', 'method_table.csv',
                                                 'pathway_tests_aki_vs_control_cameraPR_species.csv',
                                                 'redundancy_clusters_aki_vs_control.csv')],
            *[data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json' for lib in LIBRARIES]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 70, 71 and 72 first)')
HGNC_OK = all(path.is_file() for path in HGNC.values())    # G7: checked in section 4; absent -> T sections pending

NOTEBOOK_LOGIC_VERSION = '76.package_aki.1'   # Bump when a cached calculation changes.
FINGERPRINT = 'bb4a80c1b484'                   # notebook 70's confirmed AKI pass-1 labels
EXTERNAL_COMMIT = '59da04b'                    # pseudospace_reconstruction
MT_PREFIX = 'mt-'
ALPHA, NCDR_MAX, NULL_RATE = .05, .25, .05    # notebook 71's discovery level and relabeling rule
CALIBRATION_MAX = 1.5 * ALPHA                  # notebook 71: share of relabeled one-sided p <= 0.05
CAMERA_COR = .01
BASIS_DF, GRID_POINTS, DF1 = 6, 101, 6
DF2 = {'condition': 12, 'relabeling': 6}
SET_SIZES = (10, 300)
REDUNDANCY_JACCARD = .5
DETECTION = .02
# Protocol (sections 4-10), fixed before any fit
EFFECT = .25            # log2 level, drift, within-segment change; Delta mu in segment units
RATIO = .25             # profile specificity: |relabeled| < 0.25 |condition|
SEG_TOL = .5            # peak agreement, segment units
SET_CUT = ALPHA / 2     # empirical p, Bonferroni over T_PT and C
PT_ENRICH, MIN_T_PT = 1., 20
K_NN = 30
DRIFT_WINDOW = 21       # notebook 71's drift windows
LATE_Q, A1_FACTOR = .8, .75
GENE_SHARE = 2 / 3      # 5.4 B2/B3: share of the listed genes that must meet the rule (notebook 45's 2/3)
INJURY_MARKERS = ['Havcr1', 'Lcn2', 'Krt20', 'Krt8', 'Krt18', 'Spp1', 'Cd44', 'Vcam1', 'Sox9', 'Cdkn1a',
                  'Clu', 'Mt1', 'Mt2', 'Atf3', 'Hmox1', 'Timp1', 'Sprr1a', 'Tnfrsf12a']            # notebook 45
INJURY_HALLMARKS = ['TNF-alpha Signaling via NF-kB', 'p53 Pathway', 'Apoptosis', 'Hypoxia', 'Inflammatory Response',
                    'IL-6/JAK/STAT3 Signaling', 'Interferon Gamma Response', 'Epithelial Mesenchymal Transition',
                    'Unfolded Protein Response', 'mTORC1 Signaling', 'TGF-beta Signaling', 'Fatty Acid Metabolism',
                    'Oxidative Phosphorylation']                                                    # notebook 45
G2M_ID, E2F_ID = 'MSigDB_Hallmark_2020::G2-M Checkpoint', 'MSigDB_Hallmark_2020::E2F Targets'
MITOTIC = ['Mki67', 'Top2a', 'Cdk1', 'Ccnb2', 'Plk1', 'Aurkb', 'Cdc20']                             # 5.4 B2
ARREST = ['Cdkn1a', 'Gadd45a', 'Gadd45b', 'Gadd45g']                                              # 5.4 B3
DISPLAY_TRANSPORTERS = ['Slc34a1', 'Slc5a2', 'Slc5a12', 'Slc22a6', 'Slc7a13', 'Slc22a7']
DISPLAY_GENES = ['Mki67', 'Top2a', 'Havcr1', 'Lcn2']
N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', '24'))   # parallel workers; never part of a cache key

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from importlib.metadata import distribution, version
from IPython.display import display

direct_url = json.loads(distribution('pseudospace-reconstruction').read_text('direct_url.json') or '{}')
EXTERNAL = {'version': version('pseudospace-reconstruction'),
            'commit': direct_url.get('vcs_info', {}).get('commit_id', 'unknown')}
assert EXTERNAL['commit'].startswith(EXTERNAL_COMMIT), f'pseudospace_reconstruction is not commit {EXTERNAL_COMMIT}: {EXTERNAL}'

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
CONTROL_COLOR, IR_COLOR = '#2a78d6', '#b5367a'     # validated pair (light surface; CVD ΔE 16)
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


print('Results folder:', output, '· parallel workers:', N_JOBS, '· pseudospace_reconstruction', EXTERNAL['version'],
      EXTERNAL['commit'][:7], '· HGNC files present:', HGNC_OK)

### 1.1 · Inputs, guards and the mouse-panel universe

- **G1:** the export contract (`read_pt_units`); provenance notebook 70, package commit 59da04b, pass-1 fingerprint
  `bb4a80c1b484`.
- **Pool:** export genes that are mouse symbols of the one-to-one ortholog map. **Universe U76:** pool genes with
  equal-specimen detection ≥ 2% over the four AKI mice, minus the 13 mt genes. Built with two `aki_inputs` calls:
  first on the pool (mt kept) to find the detected genes, then on those genes with `exclude_genes = MT`, which builds
  notebook 37's sets directly on U76 (10–300 tested members).
- **G2:** segment medians increase S1 < S2 < S3 in every specimen. **G3:** the 13 mt genes leave the library and the
  universe.
- **G6:** c19 is pass-1 `leiden_coarse` 19 with 80 units labelled cTAL, 75 of them IR; c15 is Unassigned with 2,425
  units and no pseudospace.

In [ ]:
import h5py
from anndata.io import read_elem
from scipy import sparse

from pseudospace.package_inputs import aki_inputs, hvm_inputs, read_pt_units
from pseudospace.pathway_pipelines import partition_groups
from pseudospace.stage_cache import cached_payload, digest

t0 = time.perf_counter()
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
MOUSE_ORTHO = set(orthologs.mouse_symbol.astype(str))
MT = sorted(g for g in MOUSE_ORTHO if g.startswith(MT_PREFIX))
assert len(MT) == 13, MT
UNITS = read_pt_units(EXPORT, 'aki')                                            # G1: the contract
prov = UNITS.uns['provenance']
assert str(prov['notebook_logic_version']).startswith('70.'), prov['notebook_logic_version']
assert str(prov['package']['commit']).startswith(EXTERNAL_COMMIT), prov['package']
assert prov['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT, prov['fingerprints']['pass1']
PROVENANCE = {'notebook': str(prov['notebook']), 'logic': str(prov['notebook_logic_version']),
              'package_commit': str(prov['package']['commit']), 'pass1_fingerprint': FINGERPRINT,
              'units': int(UNITS.n_obs), 'genes': int(UNITS.n_vars)}
guard_log = ['G1: export contract; provenance notebook 70 at package 59da04b with pass-1 fingerprint ' + FINGERPRINT]
PANEL = [g for g in UNITS.var_names.astype(str) if g in MOUSE_ORTHO]
U_MT = aki_inputs(UNITS, data_root, orthologs, PANEL)['genes']                 # detected on the pool, mt kept


def h5_obs(path, columns=None):
    with h5py.File(path, 'r') as f:
        obs = read_elem(f['obs'])
    return obs if columns is None else obs[columns]


def cohort_inputs(adata, universe, title):
    """Notebook 71's AKI construction (aki_inputs) with a given candidate universe, plus the analysis keys."""
    ak = aki_inputs(adata, data_root, orthologs, universe, exclude_genes=MT)
    segment = pd.Series(ak['segment']).str.replace('PT-', '', regex=False).to_numpy()
    d = {'title': title, 'counts': ak['counts'].tocsr(), 'library': np.asarray(ak['library'], float),
         'library_with_mt': np.asarray(ak['library_with_excluded'], float), 'position': np.asarray(ak['position'], float),
         'position_raw': np.asarray(ak['position_raw'], float), 'specimen': ak['specimen'].astype(str),
         'segment': segment, 'case': np.asarray(ak['case'], bool), 'genes': list(ak['genes']), 'sets': ak['gene_sets'],
         'Y': ak['Y'].tocsr(), 'obs': ak['obs'], 'support_raw': ak['support_raw'], 'n_units': ak['n_units'],
         'mt_library': list(ak['excluded_library']), 'mt_universe': list(ak['excluded_universe'])}
    medians = pd.DataFrame({'p': d['position'], 's': d['specimen'], 'g': segment}).groupby(['s', 'g']).p.median().unstack()
    assert (medians['S1'] < medians['S2']).all() and (medians['S2'] < medians['S3']).all(), medians   # G2
    assert set(d['mt_library']) <= set(MT) and set(d['mt_universe']) <= set(MT)                        # G3
    d['partitions'] = partition_groups(d['specimen'], d['case'])
    labels = list(d['partitions'])
    assert labels == ['species', 'swap:IR2A2+Ctrl1A2', 'swap:IR2A2+Ctrl1A4'], labels
    d['short'] = dict(zip(labels, ['condition', 'relabeling 1', 'relabeling 2']))
    d['log_mean_count'] = np.log(np.asarray(d['counts'].mean(axis=0)).ravel())
    d['input_key'] = {'counts': digest(d['counts']), 'library': d['library'], 'position': d['position'],
                      'specimen': d['specimen'], 'genes': d['genes']}
    d['medians'] = medians
    return d


PRIMARY = cohort_inputs(UNITS, U_MT, 'primary (package positions, PT cohort)')
P = PRIMARY
assert len(P['mt_library']) == 13, P['mt_library']
guard_log.append(f"G2/G3 (primary): segment medians S1 < S2 < S3 in every specimen; {len(P['mt_library'])} mt genes out of "
                 f"the library, {len(P['mt_universe'])} out of the universe")

# G6: the clusters outside the PT cohort
PASS1_OBS = h5_obs(run_aki / '03_integration' / 'pass1_labelled.h5ad',
                   ['sample', 'leiden_coarse', 'seg_class', 'x_centroid', 'y_centroid'])
POSITIONS = pd.read_csv(package_aki / 'all_units_positions.csv', index_col=0)
assert POSITIONS.index.equals(PASS1_OBS.index)
cluster = PASS1_OBS.leiden_coarse.astype(str)
C19, C15 = PASS1_OBS.index[cluster.eq('19')], PASS1_OBS.index[cluster.eq('15')]
assert len(C19) == 80 and set(POSITIONS.loc[C19, 'label']) == {'cTAL'} and int(PASS1_OBS.loc[C19, 'sample'].isin(IRS).sum()) == 75
assert len(C15) == 2425 and set(POSITIONS.loc[C15, 'label']) == {'Unassigned'} and POSITIONS.loc[C15, 'pseudospace'].isna().all()
guard_log.append('G6: c19 = pass-1 cluster 19 (80 units, cTAL, 75 IR, on the curve); c15 = cluster 15 (2,425 Unassigned, no pseudospace)')

# The universe against notebook 71's
U71 = pd.read_csv(T71 / 'tables' / 'universe_aki_vs_control.csv').gene.astype(str).tolist()
SETS71 = json.loads((T71 / 'tables' / 'gene_sets_aki_vs_control.json').read_text())
shared_sets = sorted(set(SETS71) & set(P['sets']))
non_ortho_detected = None
units_det = UNITS[UNITS.obs_names.isin(P['obs'].index)]
other = [g for g in UNITS.var_names.astype(str) if g not in MOUSE_ORTHO and not g.startswith(MT_PREFIX)]
x_other = units_det[:, other].X
spec_det = units_det.obs['sample'].astype(str).to_numpy()
det_other = np.mean([np.asarray((x_other[spec_det == s] > 0).mean(axis=0)).ravel() for s in SPECIMENS], axis=0)
non_ortho_detected = int((det_other >= DETECTION).sum())
del units_det, x_other
UNIVERSE_TABLE = pd.Series({
    'export genes': UNITS.n_vars, 'pool (one-to-one ortholog panel)': len(PANEL),
    'U76 genes': len(P['genes']), "notebook 71's AKI universe": len(U71),
    "71's universe inside U76": f"{len(set(U71) & set(P['genes'])):,} (all: {set(U71) <= set(P['genes'])})",
    'genes new in U76': len(set(P['genes']) - set(U71)),
    'non-ortholog genes passing the AKI detection rule (untested)': non_ortho_detected,
    'U76 sets': len(P['sets']), "notebook 71's sets": len(SETS71), 'sets tested in both': len(shared_sets),
    'sets in both with the same members': sum(sorted(P['sets'][p]) == sorted(SETS71[p]) for p in shared_sets),
    "notebook 45's injury markers in U76 (in 71)": f"{sum(g in P['genes'] for g in INJURY_MARKERS)} "
                                                    f"({sum(g in U71 for g in INJURY_MARKERS)}) of {len(INJURY_MARKERS)}",
    'Mki67, Top2a, Pcna in U76': str([g in P['genes'] for g in ('Mki67', 'Top2a', 'Pcna')]),
    'units: export / common support': f"{P['n_units']['before support']:,} / {P['n_units']['after support']:,}",
    'raw pseudospace of the support': f"{P['support_raw'][0]:.4f}–{P['support_raw'][1]:.4f}"}, name='value')
UNIVERSE_TABLE.to_frame().to_csv(tables / 'inputs.csv')
pd.DataFrame({'gene': P['genes'], 'in_notebook_71': pd.Index(P['genes']).isin(U71)}).to_csv(tables / 'universe.csv', index=False)
(tables / 'gene_sets.json').write_text(json.dumps(P['sets'], indent=0))
display(UNIVERSE_TABLE.to_frame())
print('\n'.join('· ' + line for line in guard_log))
print(f'Inputs {time.perf_counter() - t0:.0f} s')

## 2 · The gene model and the positional F-test (notebook 71's sections 2–3, unchanged)

- **Gene model** (`standard_gene_model`): NB GLM on raw counts, offset log(mt-free library), 6-df cubic B-spline,
  prior weights n / (2·J·n_j), 101-point `common_grid`; α per gene under the condition's M_full, reused for the
  relabelings (the condition shape is their nuisance).
- **Denominator** `specimen_shape_lr`; **F** `moderated_f` (covariate log mean count, robust FALSE); gene q is BH.
- **cameraPR** on z (inter-gene correlation 0.01); a call is Up with FDR ≤ 0.05.
- **G4:** df₂ 12 / 6. **G5 (reuse, decision D2):** on notebook 71's 10,198 genes the inputs are notebook 71's (its
  inputs rebuilt by its own call and checked against its cached counts digest), and for genes converged in both runs
  α, T_spatial, LR_spec and Δ equal notebook 71's cached payloads (rtol 1e-6).

The same function runs every cohort of this notebook (primary, the two c19 arms, the package DPT).

In [ ]:
from scipy.stats import false_discovery_control

from pseudospace import nb_gam, pathway_calibration, pathway_pipelines, stats_gam
from pseudospace.pathway_pipelines import (GeneModel, camera_pr, common_grid, moderated_f, specimen_shape_lr,
                                           standard_gene_model)

cache = output / 'stage_cache'
code_key = digest([NOTEBOOK_LOGIC_VERSION] + [Path(m.__file__) for m in (nb_gam, pathway_calibration, pathway_pipelines, stats_gam)])
SECONDS = {}


def cached(stage, compute, *, params=None, inputs=None):
    def wrapped():
        t = time.perf_counter()
        out = dict(compute())
        out['__seconds'] = np.array(time.perf_counter() - t)
        return out
    result = cached_payload(stage, wrapped, root=cache, params=params, inputs=inputs, code=code_key)
    SECONDS[stage] = float(result.pop('__seconds'))
    return result


def model_payload(model):
    out = {f'stat|{c}': model.statistics[c].to_numpy() for c in model.statistics}
    out.update(grid=model.grid, knots=model.knots, reference=model.reference, case=model.case, delta=model.delta,
               se=model.se, group_names=model.groups.index.to_numpy().astype(str), group_values=model.groups.to_numpy(),
               alpha=np.asarray(model.dispersion['alpha'], float), alpha_converged=np.asarray(model.dispersion['converged'], bool))
    for name, curve in model.specimen_curves.items():
        out[f'curve|{name}'] = curve
        out[f'converged|{name}'] = model.specimen_converged[name].to_numpy()
        out[f'separated|{name}'] = model.specimen_separated[name].to_numpy()
    return out


def model_from(payload, genes):
    index = pd.Index(genes, name='gene')
    stats = pd.DataFrame({c: payload[f'stat|{c}'] for c in ('T_level', 'T_spatial', 'T_total', 'alpha', 'converged',
                                                            'separated')}, index=index)
    stats[['converged', 'separated']] = stats[['converged', 'separated']].astype(bool)
    names = sorted(k.split('|', 1)[1] for k in payload if k.startswith('curve|'))
    return GeneModel(statistics=stats, grid=payload['grid'], knots=payload['knots'], reference=payload['reference'],
                     case=payload['case'], delta=payload['delta'], se=payload['se'],
                     specimen_curves={n: payload[f'curve|{n}'] for n in names},
                     specimen_converged=pd.DataFrame({n: payload[f'converged|{n}'].astype(bool) for n in names}, index=index),
                     specimen_separated=pd.DataFrame({n: payload[f'separated|{n}'].astype(bool) for n in names}, index=index),
                     groups=pd.Series(payload['group_values'].astype(int), index=payload['group_names'].astype(str)),
                     dispersion={'alpha': payload['alpha'], 'converged': payload['alpha_converged'].astype(bool)})


def stored(folder, stage):
    """Another notebook's newest cached payload of one stage (read only)."""
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}, files[-1]


def slug(label):
    return label.replace(':', '_').replace('+', '_')


def fit_cohort(tag, d):
    """Gene models and denominators of the three partitions, then the positional F-test (notebook 71, unchanged)."""
    models, denom, stat = {}, {}, {}
    for label, (group, nuisance) in d['partitions'].items():
        dispersion = None if label == 'species' else models['species'].dispersion
        payload = cached(f'gene_model_{tag}_{slug(label)}', lambda group=group, nuisance=nuisance, dispersion=dispersion:
                         model_payload(standard_gene_model(d['counts'], d['library'], d['position'], group, d['specimen'],
                                                           genes=d['genes'], nuisance_shape=nuisance, dispersion=dispersion,
                                                           basis_df=BASIS_DF, n_jobs=N_JOBS)),
                         params={'basis_df': BASIS_DF, 'grid_points': GRID_POINTS, 'label': label},
                         inputs={**d['input_key'], 'group': group, 'nuisance': nuisance,
                                 'alpha': None if dispersion is None else dispersion['alpha']})
        models[label] = model_from(payload, d['genes'])
        alpha = models['species'].dispersion
        denom[label] = cached(f'specimen_shape_{tag}_{slug(label)}',
                              lambda group=group, nuisance=nuisance, alpha=alpha:
                              specimen_shape_lr(d['counts'], d['library'], d['position'], group, d['specimen'], alpha,
                                                nuisance_shape=nuisance, basis_df=BASIS_DF, n_jobs=N_JOBS),
                              params={'basis_df': BASIS_DF, 'label': label},
                              inputs={**d['input_key'], 'group': group, 'nuisance': nuisance, 'alpha': alpha['alpha'],
                                      'alpha_converged': alpha['converged']})
        expected = DF2['condition' if label == 'species' else 'relabeling']
        assert int(denom[label]['df2']) == expected, (tag, label, int(denom[label]['df2']))          # G4
        model = models[label]
        assert model.delta.shape == (len(d['genes']), GRID_POINTS), model.delta.shape
        frame = moderated_f(model.statistics.T_spatial, DF1, np.asarray(denom[label]['LR_spec'], float),
                            int(denom[label]['df2']), covariate=d['log_mean_count'], robust=False)
        frame['q'] = false_discovery_control(frame.p.to_numpy())
        frame['converged'] = model.statistics.converged.to_numpy() & np.asarray(denom[label]['converged'], bool)
        frame['separated'] = model.statistics.separated.to_numpy() | np.asarray(denom[label]['separated'], bool)
        frame['T_spatial'] = model.statistics.T_spatial.to_numpy()
        frame['LR_spec'] = np.asarray(denom[label]['LR_spec'], float)
        assert np.isfinite(frame.z).all()
        stat[label] = frame
    condition = models['species']
    np.testing.assert_allclose(condition.grid, common_grid(d['position'], d['specimen'], GRID_POINTS), rtol=0, atol=1e-12)
    for model in models.values():
        np.testing.assert_array_equal(model.grid, condition.grid)
    return models, denom, stat


t0 = time.perf_counter()
MODELS, DENOM, STAT = {}, {}, {}
MODELS['primary'], DENOM['primary'], STAT['primary'] = fit_cohort('primary', P)
guard_log.append('G4 (primary): df2 = 12 (condition) and 6 (each relabeling); grids equal common_grid')

# G5: notebook 71's inputs, rebuilt by its own call, and its cached fits
hvm_units = read_pt_units(results_root / 'package_pseudospace' / 'hvm' / 'pt_units.h5ad', 'hvm')
kept_h = hvm_inputs(hvm_units, data_root, orthologs)                          # mt kept: notebook 71's AKI pool
del hvm_units
ak71 = aki_inputs(UNITS, data_root, orthologs, kept_h['genes'], exclude_genes=MT)
assert ak71['genes'] == U71, "notebook 71's AKI universe is not reproduced by its own call"
_, key_file = stored(T71, 'gene_model_aki_vs_control_species')
key71 = re.search(r"'counts': '([0-9a-f]{16})'", json.loads(Path(str(key_file) + '.key.json').read_text())['inputs']).group(1)
assert digest(ak71['counts'].tocsr()) == key71, "the rebuilt notebook 71 counts do not match its cached digest"
idx71 = pd.Index(P['genes']).get_indexer(U71)
assert (idx71 >= 0).all()
assert (P['counts'][:, idx71] != ak71['counts'].tocsr()).nnz == 0
np.testing.assert_array_equal(P['library'], ak71['library'])
np.testing.assert_array_equal(P['position'], ak71['position'])
del kept_h, ak71
REUSE_ROWS = []
for label in P['partitions']:
    p71, _ = stored(T71, f'gene_model_aki_vs_control_{slug(label)}')
    d71, _ = stored(T71, f'specimen_shape_aki_vs_control_{slug(label)}')
    mine, den = MODELS['primary'][label], DENOM['primary'][label]
    both = (p71['stat|converged'].astype(bool) & ~p71['stat|separated'].astype(bool) & d71['converged'].astype(bool)
            & mine.statistics.converged.to_numpy()[idx71] & ~mine.statistics.separated.to_numpy()[idx71]
            & np.asarray(den['converged'], bool)[idx71])
    checks = {'alpha': (mine.statistics.alpha.to_numpy()[idx71], p71['stat|alpha']),
              'T_spatial': (mine.statistics.T_spatial.to_numpy()[idx71], p71['stat|T_spatial']),
              'LR_spec': (np.asarray(den['LR_spec'], float)[idx71], d71['LR_spec']),
              'delta': (mine.delta[idx71], p71['delta'])}
    for name, (a, b) in checks.items():
        a, b = np.asarray(a, float)[both], np.asarray(b, float)[both]
        close = np.isclose(a, b, rtol=1e-6, atol=1e-9)
        bad = ~close.all(axis=1) if close.ndim > 1 else ~close
        genes_bad = list(np.asarray(U71)[both][bad])
        REUSE_ROWS.append({'partition': P['short'][label], 'quantity': name, 'genes compared (converged in both)': int(both.sum()),
                           'genes outside rtol 1e-6': len(genes_bad), 'which': ', '.join(genes_bad),
                           'max |difference|': float(np.max(np.abs(a - b))) if len(a) else np.nan,
                           'max relative difference of those genes': float(np.max(np.abs(a - b)[bad] / np.abs(b)[bad])) if len(genes_bad) else 0.})
        # Reading (section 14): the per-gene alpha is an optimiser output; a relative difference up to 1e-4 is recorded,
        # not a stop, provided T_spatial, LR_spec and Delta of every gene agree at rtol 1e-6.
        if name == 'alpha':
            np.testing.assert_allclose(a, b, rtol=1e-4, atol=1e-9, err_msg=f'G5 {label} alpha beyond 1e-4')
        else:
            np.testing.assert_allclose(a, b, rtol=1e-6, atol=1e-9, err_msg=f'G5 {label} {name}')
REUSE = pd.DataFrame(REUSE_ROWS)
REUSE.to_csv(tables / 'reuse_guard.csv', index=False)
guard_log.append(f"G5: notebook 71's AKI inputs reproduced (counts digest {key71}); its fits equal ours on converged genes (rtol 1e-6)")
display(REUSE)
print('\n'.join('· ' + line for line in guard_log))
print(f'Gene models {time.perf_counter() - t0:.0f} s')

## 3 · The screen: pass rule, gene-level calibration and comparison with notebook 71

**Pass rule (notebook 71):** NCDR < 0.25; each relabeling calls < 5% of tested sets; calibration share (pooled
relabeled one-sided p ≤ 0.05) ≤ 0.075. Also the gene-level calibration share (pooled relabeled gene p ≤ 0.05).
**Labelled:** these calls are the Figure 6 universe's (notebook 76), not the method-comparison record (notebook 71).

In [ ]:
from pseudospace_reconstruction.compare.pathways import cluster_gene_sets
from pseudospace_reconstruction.io.gene_sets import build_gene_sets

RECORD_NOTE = 'Figure 6 universe (notebook 76); not the method-comparison record'


def screen(tag, d, stat):
    tests = {label: camera_pr(stat[label].z, d['sets'], inter_gene_cor=CAMERA_COR, use_ranks=False, fdr=ALPHA)
             for label in d['partitions']}
    labels = list(d['partitions'])
    calls = {label: set(t.pathway_id[t.call.astype(bool)]) for label, t in tests.items()}
    n_rel = [len(calls[label]) for label in labels[1:]]
    tested = len(d['sets'])
    ncdr = np.mean(n_rel) / len(calls['species']) if calls['species'] else np.nan
    p_rel = np.concatenate([tests[label].p_one_sided.to_numpy(float) for label in labels[1:]])
    gene_rel = np.concatenate([stat[label].p.to_numpy(float) for label in labels[1:]])
    row = {'cohort': tag, 'tested sets': tested, 'calls': len(calls['species']), 'relabeled calls': ', '.join(map(str, n_rel)),
           'NCDR': ncdr, 'relabeling rule': 'pass' if calls['species'] and ncdr < NCDR_MAX and max(n_rel) < NULL_RATE * tested else 'fail',
           'calibration share': float(np.mean(p_rel <= ALPHA)),
           'calibration': 'pass' if np.mean(p_rel <= ALPHA) <= CALIBRATION_MAX else 'fail',
           'genes q ≤ 0.05 (condition)': int((stat['species'].q <= ALPHA).sum()),
           'gene-level calibration share': float(np.mean(gene_rel <= ALPHA)), 'note': RECORD_NOTE}
    return tests, calls, row


TESTS, CALLS, PASS_ROWS = {}, {}, []
TESTS['primary'], CALLS['primary'], row = screen('primary', P, STAT['primary'])
PASS_ROWS.append(row)
for label, test in TESTS['primary'].items():
    test.assign(partition=P['short'][label], note=RECORD_NOTE).to_csv(tables / f"pathway_tests_{P['short'][label].replace(' ', '_')}.csv", index=False)
pd.concat({P['short'][label]: STAT['primary'][label] for label in P['partitions']}, axis=1).to_csv(tables / 'gene_statistics.csv')
M71 = pd.read_csv(T71 / 'tables' / 'method_table.csv')
row71 = M71[M71.dataset.eq('AKI vs control mouse PT') & M71.method.eq('normalized · cameraPR on z')].iloc[0]
PASS_ROWS.append({'cohort': 'notebook 71 (record)', 'tested sets': len(SETS71), 'calls': int(row71['calls']),
                  'relabeled calls': row71['relabeled calls'], 'NCDR': row71['NCDR'], 'relabeling rule': row71['relabeling rule'],
                  'calibration share': row71['calibration share'], 'calibration': row71['calibration'],
                  'note': 'method-comparison record'})

# Redundancy clusters on U76's sets (notebook 71's rule and guard G6 there)
sets_pkg = build_gene_sets({lib: {p.split('::', 1)[1]: v for p, v in P['sets'].items() if p.startswith(lib + '::')}
                            for lib in LIBRARIES}, P['genes'], min_genes=SET_SIZES[0], max_genes=SET_SIZES[1])
ids = [f'{lib}::{name}' for lib, name in zip(sets_pkg.library, sets_pkg.pathway)]
assert sorted(ids) == sorted(P['sets'])
clusters = cluster_gene_sets(sets_pkg.membership(), sets_pkg.names, REDUNDANCY_JACCARD)
clusters.index = pd.Index(ids, name='pathway_id')
medoid = clusters[clusters.is_representative].reset_index().set_index('cluster_id').pathway_id
clusters['representative'] = clusters.cluster_id.map(medoid)
REDUNDANCY = clusters.reindex(list(P['sets']))
REDUNDANCY.to_csv(tables / 'redundancy_clusters.csv')

# Comparison with notebook 71 (genes and calls)
G71 = pd.read_csv(T71 / 'tables' / 'gene_statistics_aki_vs_control.csv', header=[0, 1], index_col=0)
C71 = pd.read_csv(T71 / 'tables' / 'pathway_tests_aki_vs_control_cameraPR_species.csv')
R71 = pd.read_csv(T71 / 'tables' / 'redundancy_clusters_aki_vs_control.csv', index_col=0)
calls71 = set(C71.pathway_id[C71.call.astype(bool)]) & set(shared_sets)
calls76 = CALLS['primary']['species'] & set(shared_sets)
rep71 = lambda calls: set(R71.representative.reindex(sorted(calls)))          # noqa: E731
from scipy.stats import spearmanr

COMPARE71 = pd.Series({
    'genes in both': len(U71),
    'Spearman of condition z (genes in both)': spearmanr(STAT['primary']['species'].z.reindex(U71), G71[('condition', 'z')].reindex(U71)).statistic,
    'Spearman of relabeling-1 z': spearmanr(STAT['primary']['swap:IR2A2+Ctrl1A2'].z.reindex(U71), G71[('relabeling 1', 'z')].reindex(U71)).statistic,
    'sets tested in both': len(shared_sets), 'calls among them: 71 / 76': f'{len(calls71)} / {len(calls76)}',
    'kept / lost / gained': f'{len(calls71 & calls76)} / {len(calls71 - calls76)} / {len(calls76 - calls71)}',
    'Jaccard over sets': len(calls71 & calls76) / max(len(calls71 | calls76), 1),
    "Jaccard over redundancy clusters (71's clusters)": len(rep71(calls71) & rep71(calls76)) / max(len(rep71(calls71) | rep71(calls76)), 1),
    'calls on sets tested only in 76': len(CALLS['primary']['species'] - set(shared_sets)),
    'redundancy clusters touched by 76 calls': len(set(REDUNDANCY.representative.reindex(sorted(CALLS['primary']['species']))))},
    name='value')
COMPARE71.to_frame().to_csv(tables / 'compare71.csv')
PASS_TABLE = pd.DataFrame(PASS_ROWS)
PASS_TABLE.to_csv(tables / 'pass_rule.csv', index=False)
display(PASS_TABLE.round(3))
display(COMPARE71.to_frame())
print('Calls only on U76-only sets:', sorted(CALLS['primary']['species'] - set(shared_sets))[:20])

## 4 · The programs (fixed in the protocol)

- **T:** notebook 74's set: HGNC "Solute carriers" (752) and "ATP binding cassette transporters" (417) branches,
  protein-coding, mapped by human symbol through the ortholog map into U76. sha256 asserted (G7).
- **T_PT (primary transport program):** T without SLC25, restricted to genes PT-enriched in **both control mice**:
  log₂(rate_PT / rate_nonPT) ≥ 1, rate = Σcounts / Σ(mt-free library) over that control's pass-2 units (PT-S1/S2/S3
  against all its other pass-2 units). **The selection uses no IR unit and no IR − control difference, so it is
  independent of the IR contrast**; it shares only the control PT rate with the contrast (about 2,700 PT units per
  control). Fewer than 20 genes would make T without SLC25 primary.
- **C (cell-cycle program):** Hallmark G2-M Checkpoint ∪ E2F Targets in U76; also G2-M and E2F separately, and the
  5.4 parts G2M′ (G2-M only) and E2F′ (E2F only, without Cdkn1a).
- **R:** notebook 45's 18 injury markers.

In [ ]:
import hashlib

PROGRAMS, PROGRAM_ROWS = {}, []
T_PENDING = None
if HGNC_OK:
    for gid, path in HGNC.items():
        assert hashlib.sha256(path.read_bytes()).hexdigest() == HGNC_SHA[gid], f'G7: HGNC {gid} sha256 differs'
    hgnc = pd.concat([pd.read_csv(HGNC[g], sep='\t').assign(branch=g) for g in HGNC], ignore_index=True)
    hgnc = hgnc[hgnc['Locus type'].eq('gene with protein product')]
    hgnc = hgnc.drop_duplicates('Approved symbol')
    human_to_mouse = orthologs.assign(h=orthologs.human_symbol.astype(str).str.upper()).drop_duplicates('h').set_index('h').mouse_symbol.astype(str)
    hgnc['mouse'] = hgnc['Approved symbol'].astype(str).str.upper().map(human_to_mouse)
    hgnc['slc25'] = hgnc['Approved symbol'].astype(str).str.upper().str.startswith('SLC25')
    in_u = hgnc[hgnc.mouse.isin(P['genes'])]
    PROGRAMS['T'] = sorted(in_u.mouse)
    guard_log.append('G7: HGNC branch files match notebook 74 sha256')
else:
    T_PENDING = 'HGNC files absent: T, T_PT and claims A3, A5 pending'
    print(T_PENDING)

# Control PT enrichment (pass-2 units of the control mice only)
with h5py.File(run_aki / '02_qc' / 'units_qc.h5ad', 'r') as f:
    QC_COUNTS = read_elem(f['layers/counts']).tocsr()
    qc_index = pd.Index(read_elem(f['obs'][f['obs'].attrs['_index']]).astype(str))
    qc_var = pd.Index(read_elem(f['var'][f['var'].attrs['_index']]).astype(str))
assert qc_index.equals(POSITIONS.index) and qc_var.equals(UNITS.var_names)
lib_cols = qc_var.isin(MOUSE_ORTHO) & ~qc_var.isin(MT)
qc_library = np.asarray(QC_COUNTS[:, lib_cols].sum(axis=1)).ravel()
on_curve = POSITIONS.pseudospace.notna().to_numpy()
is_pt = POSITIONS.label.isin(['PT-S1', 'PT-S2', 'PT-S3']).to_numpy()
ENRICH = {}
if 'T' in PROGRAMS:
    cols = qc_var.get_indexer(PROGRAMS['T'])
    for s in CONTROLS:
        in_s = on_curve & POSITIONS['sample'].eq(s).to_numpy()
        a, b = in_s & is_pt, in_s & ~is_pt
        rate_pt = np.asarray(QC_COUNTS[a][:, cols].sum(axis=0)).ravel() / qc_library[a].sum()
        rate_non = np.asarray(QC_COUNTS[b][:, cols].sum(axis=0)).ravel() / qc_library[b].sum()
        with np.errstate(divide='ignore', invalid='ignore'):
            ENRICH[s] = np.log2(rate_pt / rate_non)
    enrich = pd.DataFrame(ENRICH, index=PROGRAMS['T'])
    slc25 = set(in_u.mouse[in_u.slc25])
    enriched = enrich.ge(PT_ENRICH).all(axis=1)
    PROGRAMS['T_PT'] = sorted(g for g in PROGRAMS['T'] if enriched[g] and g not in slc25)
    T_PT_RULE = 'PT-enriched SLC (without SLC25) and ABC'
    if len(PROGRAMS['T_PT']) < MIN_T_PT:
        PROGRAMS['T_PT'] = sorted(g for g in PROGRAMS['T'] if g not in slc25)
        T_PT_RULE = 'fallback: T without SLC25 (fewer than 20 PT-enriched)'
    T_TABLE = in_u.set_index('mouse')[['Approved symbol', 'branch', 'slc25']].join(enrich.add_prefix('log2 PT/non-PT ')) \
        .assign(in_T_PT=lambda t: t.index.isin(PROGRAMS['T_PT']))
    T_TABLE.to_csv(tables / 'programs_transporters.csv')
g2m, e2f = set(P['sets'][G2M_ID]), set(P['sets'][E2F_ID])
PROGRAMS['C'] = sorted(g2m | e2f)
PROGRAMS['G2-M'], PROGRAMS['E2F'] = sorted(g2m), sorted(e2f)
PROGRAMS["G2M'"] = sorted(g2m - e2f)
PROGRAMS["E2F'"] = sorted((e2f - g2m) - {'Cdkn1a'})
PROGRAMS['R'] = [g for g in INJURY_MARKERS if g in P['genes']]
SIGN = {'T': -1, 'T_PT': -1}        # loss programs are summarised as L = -T
NAME = {'T': 'loss T', 'T_PT': 'loss T_PT'}
PROGRAM_TABLE = pd.DataFrame([{'program': k, 'members in U76': len(v), 'summarised as': NAME.get(k, k),
                               'members': ', '.join(v) if len(v) <= 60 else ', '.join(v[:60]) + ', …'} for k, v in PROGRAMS.items()])
PROGRAM_TABLE.to_csv(tables / 'programs.csv', index=False)
display(PROGRAM_TABLE[['program', 'members in U76', 'summarised as']])
if 'T' in PROGRAMS:
    print(f"T: {len(PROGRAMS['T'])} genes in U76 ({int(in_u.slc25.sum())} SLC25, {int(in_u.branch.eq(417).sum())} ABC); "
          f"T_PT rule: {T_PT_RULE}; {len(PROGRAMS['T_PT'])} genes:", ', '.join(PROGRAMS['T_PT']))

## 5 · Segment units and program profiles

- **Segment units:** t₁₂ and t₂₃ maximise balanced accuracy of S1 vs S2 and S2 vs S3 on the analysed position, using
  control units only; [0, t₁₂] → [0, 1], [t₁₂, t₂₃] → [1, 2], [t₂₃, 1] → [2, 3], extended linearly.
- **Δ_g(x)** is the natural-log rate ratio IR/control between equal-specimen group curves; **pair profiles** use the
  specimen curves of the 4 IR × control pairs; **relabeled profiles** the relabelings' Δ.
- **Centring:** minus the median over converged, non-separated genes at each x (each contrast its own median).
- **Program profile:** median over members of the centred Δ, in log₂; loss programs are summarised as L = −T.
- **Summaries:** level (grid mean), drift (last 21 − first 21 grid points), centre of mass of the positive part (μ,
  segment units), peak, and the change across each segment's stretch.
- **Discrete view:** the same program medians on centred log₂ IR/control ratios of specimen pseudobulks (whole PT,
  S1, S2, S3); a pseudobulk rate is floored at 0.5 count over its summed library (notebook 71's floor).

In [ ]:
from pseudospace.aki_along_pt import (centre_rows, empirical_p, equal_weight_quantiles, knn_place, profile_summary,
                                      program_profile, segment_cuts, segment_units)

PAIRS = [(i, c) for i in IRS for c in CONTROLS]
PAIR_NAMES = [f'{i}−{c}' for i, c in PAIRS]


def segment_scale(d):
    ctrl = np.isin(d['specimen'], CONTROLS)
    return segment_cuts(d['position'][ctrl], 'PT-' + pd.Series(d['segment'][ctrl]).astype(str))


def contrasts(d, models):
    """{contrast: (centred genes x grid, usable mask, m(x))} for the condition, the relabelings and the 4 pairs."""
    out = {}
    cond = models['species']
    ok = (cond.statistics.converged & ~cond.statistics.separated).to_numpy()
    c, m = centre_rows(cond.delta, ok)
    out['condition'] = (c, ok, m)
    for label in list(d['partitions'])[1:]:
        model = models[label]
        ok_r = (model.statistics.converged & ~model.statistics.separated).to_numpy()
        c, m = centre_rows(model.delta, ok_r)
        out[d['short'][label]] = (c, ok_r, m)
    for (i, k), name in zip(PAIRS, PAIR_NAMES):
        ok_p = (ok & cond.specimen_converged[i].to_numpy() & cond.specimen_converged[k].to_numpy()
                & ~cond.specimen_separated[i].to_numpy() & ~cond.specimen_separated[k].to_numpy())
        c, m = centre_rows(cond.specimen_curves[i] - cond.specimen_curves[k], ok_p)
        out[name] = (c, ok_p, m)
    return out


def profiles_of(d, models, programs):
    t12, t23 = segment_scale(d)
    grid = models['species'].grid
    s = segment_units(grid, t12, t23)
    con = contrasts(d, models)
    prof, rows = {}, []
    for cname, (c, ok, _) in con.items():
        for pname, members in programs.items():
            profile, n = program_profile(c, d['genes'], members, usable=ok)
            signed = SIGN.get(pname, 1) * profile
            prof[(cname, NAME.get(pname, pname))] = signed
            if n:
                rows.append({'contrast': cname, 'program': NAME.get(pname, pname), 'members used': n,
                             **profile_summary(signed, s, window=DRIFT_WINDOW)})
    summ = pd.DataFrame(rows).set_index(['contrast', 'program'])
    return {'t12': t12, 't23': t23, 'grid': grid, 's': s, 'contrasts': con, 'profiles': prof, 'summary': summ}


def program_tests(d, stat, programs, tests):
    """cameraPR of each program on each partition's z; p_emp against the pooled relabeled one-sided p of all sets."""
    labels = list(d['partitions'])
    null = np.concatenate([tests[label].p_one_sided.to_numpy(float) for label in labels[1:]])
    rows = []
    for pname, members in programs.items():
        present = [g for g in members if g in set(d['genes'])]
        if len(present) < 2:
            continue
        row = {'program': NAME.get(pname, pname), 'n': len(present)}
        for label in labels:
            t = camera_pr(stat[label].z, {pname: present}, inter_gene_cor=CAMERA_COR, use_ranks=False, fdr=ALPHA).iloc[0]
            row[f"{d['short'][label]} direction"] = t.direction
            row[f"{d['short'][label]} p one-sided"] = float(t.p_one_sided)
        row['p_emp (condition)'] = float(empirical_p(row['condition p one-sided'], null))
        row['own relabeled p > 0.05 (both)'] = all(row[f'relabeling {k} p one-sided'] > ALPHA for k in (1, 2))
        rows.append(row)
    return pd.DataFrame(rows).set_index('program')


def discrete_view(d, programs):
    """Program medians of centred log2 IR/control ratios of specimen pseudobulks: whole PT and each segment."""
    out = {}
    counts, lib = d['counts'], d['library']
    for part in ('whole PT', 'S1', 'S2', 'S3'):
        rates = {}
        for spec in SPECIMENS:
            mask = (d['specimen'] == spec) & ((d['segment'] == part) if part != 'whole PT' else True)
            total = lib[mask].sum()
            rates[spec] = np.maximum(np.asarray(counts[mask].sum(axis=0)).ravel(), .5) / total
        log = pd.DataFrame({k: np.log2(v) for k, v in rates.items()}, index=d['genes'])
        diff = log[IRS].mean(axis=1) - log[CONTROLS].mean(axis=1)
        diff -= diff.median()
        for pname, members in programs.items():
            present = [g for g in members if g in diff.index]
            if present:
                out[(part, NAME.get(pname, pname))] = SIGN.get(pname, 1) * float(diff[present].median())
    return pd.Series(out).unstack(0)[['whole PT', 'S1', 'S2', 'S3']]


PROF, PTEST, DISCRETE = {}, {}, {}
PROF['primary'] = profiles_of(P, MODELS['primary'], PROGRAMS)
PTEST['primary'] = program_tests(P, STAT['primary'], PROGRAMS, TESTS['primary'])
DISCRETE['primary'] = discrete_view(P, PROGRAMS)
pr = PROF['primary']
SEGMENT_UNITS = pd.DataFrame({'grid position': pr['grid'], 'segment units': pr['s'],
                              'm(x) condition (ln)': pr['contrasts']['condition'][2]})
SEGMENT_UNITS.to_csv(tables / 'segment_units.csv', index=False)
np.savez_compressed(tables / 'profiles.npz', grid=pr['grid'], s=pr['s'],
                    **{f'{c}|{p}': v for (c, p), v in pr['profiles'].items()},
                    **{f'm|{c}': v[2] for c, v in pr['contrasts'].items()})
pr['summary'].to_csv(tables / 'profile_summaries.csv')
PTEST['primary'].to_csv(tables / 'program_tests.csv')
DISCRETE['primary'].to_csv(tables / 'discrete_view.csv')
print(f"Segment cuts on the analysed position: t12 = {pr['t12']:.3f}, t23 = {pr['t23']:.3f}; the grid spans "
      f"{pr['s'][0]:.2f}–{pr['s'][-1]:.2f} segment units; m(x) ranges {pr['contrasts']['condition'][2].min():+.3f} to "
      f"{pr['contrasts']['condition'][2].max():+.3f} (ln)")
display(pr['summary'].xs('condition', level='contrast').round(3))
display(PTEST['primary'].round(4))
display(DISCRETE['primary'].round(3))

## 6 · (b) The positive control: notebook 45's D6 on the package axis

- **Markers:** for each R marker in U76, the IR − control difference of equal-specimen segment means of Y
  (log-normalised, full library); the share that rises most in S3 (notebook 45: 13 of 14 on the mouse-only PT DPT).
  Continuous version: the share whose argmax of Δ_g(x) lies at ≥ 2 segment units.
- **Hallmarks:** notebook 45's 13 injury Hallmarks called by the screen; Mann–Whitney (one-sided) that their
  one-sided p rank below the other sets'.
- **Location:** the share of called sets whose members' median IR − control segment difference is largest in
  absolute value in S3 (notebook 45's rule).
- **Detection passes** if ≥ 2/3 of markers rise most in S3 and (≥ 3 Hallmarks called or Mann–Whitney p ≤ 0.01).
  **Specificity** is reported under notebook 45's rule (mean relabeled calls ≤ 5% of condition calls) and the R4
  rule (NCDR < 0.25 and each relabeling < 5% of tested sets).

In [ ]:
from scipy.stats import mannwhitneyu


def segment_means(d):
    """Equal-specimen IR - control differences of segment means of Y: genes x S1/S2/S3."""
    y = d['Y']
    out = {}
    for seg in ('S1', 'S2', 'S3'):
        means = {}
        for spec in SPECIMENS:
            mask = (d['specimen'] == spec) & (d['segment'] == seg)
            means[spec] = np.asarray(y[mask].mean(axis=0)).ravel()
        out[seg] = np.mean([means[s] for s in IRS], axis=0) - np.mean([means[s] for s in CONTROLS], axis=0)
    return pd.DataFrame(out, index=d['genes'])


def positive_control(d, models, prof, tests, calls):
    member_delta = segment_means(d)
    markers = [g for g in INJURY_MARKERS if g in d['genes']]
    seg = member_delta.loc[markers]
    rises = seg.idxmax(axis=1)
    cond = models['species']
    rows = pd.Index(d['genes']).get_indexer(markers)
    argmax_s = prof['s'][np.argmax(cond.delta[rows], axis=1)]
    t = tests['species'].set_index('pathway_id')
    hall = [f'MSigDB_Hallmark_2020::{h}' for h in INJURY_HALLMARKS if f'MSigDB_Hallmark_2020::{h}' in t.index]
    mw = mannwhitneyu(t.loc[hall, 'p_one_sided'], t.drop(index=hall).p_one_sided, alternative='less').pvalue
    called = sorted(calls['species'])
    located = pd.Series({p: member_delta.loc[d['sets'][p]].median().abs().idxmax() for p in called}, dtype=object)
    labels = list(d['partitions'])
    n_rel = [len(calls[label]) for label in labels[1:]]
    n_cond = len(calls['species'])
    hall_called = [h.split('::')[1] for h in hall if h in calls['species']]
    share = float(rises.eq('S3').mean())
    detection = share >= 2 / 3 and (len(hall_called) >= 3 or mw <= .01)
    summary = pd.Series({
        'injury markers tested': len(markers), 'rising most in S3 (segment means)': int(rises.eq('S3').sum()),
        'share in S3': share, 'continuous: argmax at ≥ 2 segment units': int((argmax_s >= 2).sum()),
        'injury Hallmarks tested': len(hall), 'injury Hallmarks called': len(hall_called),
        'called injury Hallmarks': '; '.join(hall_called), 'Mann–Whitney p (injury Hallmarks rank higher)': mw,
        'condition calls': n_cond, 'called sets with largest change in S3': float(located.eq('S3').mean()) if len(located) else np.nan,
        'detection': 'pass' if detection else 'fail',
        "specificity, notebook 45's rule (mean relabeled ≤ 5% of condition calls)":
            'pass' if n_cond and np.mean(n_rel) <= NULL_RATE * n_cond else 'fail',
        'specificity, R4 rule': 'pass' if n_cond and np.mean(n_rel) / n_cond < NCDR_MAX and max(n_rel) < NULL_RATE * len(d['sets']) else 'fail'},
        name='value')
    table = seg.assign(**{'largest increase': rises, 'argmax of Δ (segment units)': argmax_s})
    return summary, table, detection


POSCTRL = {}
POSCTRL['primary'] = positive_control(P, MODELS['primary'], PROF['primary'], TESTS['primary'], CALLS['primary'])
POSCTRL['primary'][0].to_frame().to_csv(tables / 'positive_control.csv')
POSCTRL['primary'][1].to_csv(tables / 'positive_control_markers.csv')
display(POSCTRL['primary'][0].to_frame())
display(POSCTRL['primary'][1].round(3))

## 7 · Placement of the units without a PT position (protocol section 8)

- **Neighbours:** k = 30 in pass-1 `X_harmony` (Euclidean). For c19 and c15 the reference is every unit with a
  pseudospace **except c19** (otherwise c19 units find each other). Each placed unit gets the PT share of its 30
  neighbours, their median raw pseudospace and its `seg_class`.
- **On the PT stretch:** PT share ≥ 0.5 and median ≤ c (notebook 70's pooled PT cut). Counts per specimen and per
  placed segment (majority label among its PT neighbours).
- **Arm (ii) of the c19 sensitivity:** each c19 unit at the median raw pseudospace of its 30 nearest PT units, with
  their majority label.

In [ ]:
with h5py.File(run_aki / '03_integration' / 'pass1_labelled.h5ad', 'r') as f:
    HARMONY = np.asarray(read_elem(f['obsm/X_harmony']), float)
assert HARMONY.shape == (len(POSITIONS), 50)
STRETCH = pd.read_csv(package_aki / 'tables' / 'pt_stretch.csv').set_index('specimen')
CUT = float(STRETCH.loc['pooled', 'cut'])
row_of = pd.Series(np.arange(len(POSITIONS)), index=POSITIONS.index)
ref_mask = on_curve & ~POSITIONS.index.isin(C19)
labels_all = POSITIONS.label.astype(str).to_numpy()
PLACE_ROWS = []
PLACED = {}
for name, ids in (('c19', C19), ('c15', C15)):
    q = HARMONY[row_of[ids].to_numpy()]
    near = knn_place(q, HARMONY[ref_mask], POSITIONS.pseudospace.to_numpy()[ref_mask], labels_all[ref_mask], k=K_NN)
    pt_cols = [c for c in near if c in ('share_PT-S1', 'share_PT-S2', 'share_PT-S3')]
    near['pt_share'] = near[pt_cols].sum(axis=1)
    near['pt_majority'] = near[pt_cols].idxmax(axis=1).str.replace('share_', '', regex=False)
    near.index = ids
    near['sample'] = PASS1_OBS.loc[ids, 'sample'].astype(str).to_numpy()
    near['seg_class'] = PASS1_OBS.loc[ids, 'seg_class'].astype(str).to_numpy()
    near['on_pt_stretch'] = near.pt_share.ge(.5) & near.median_value.le(CUT)
    PLACED[name] = near
pt_ref = on_curve & is_pt
onto_pt = knn_place(HARMONY[row_of[C19].to_numpy()], HARMONY[pt_ref], POSITIONS.pseudospace.to_numpy()[pt_ref],
                    labels_all[pt_ref], k=K_NN)
onto_pt.index = C19
PLACED['c19 onto PT'] = onto_pt
for name in ('c19', 'c15'):
    near = PLACED[name]
    for spec in SPECIMENS:
        sub = near[near['sample'].eq(spec)]
        on = sub[sub.on_pt_stretch]
        PLACE_ROWS.append({'cluster': name, 'specimen': spec, 'units': len(sub), 'on the PT stretch': len(on),
                           **{f'on stretch, placed {k}': int(on.pt_majority.eq(k).sum()) for k in ('PT-S1', 'PT-S2', 'PT-S3')},
                           'median PT share': float(sub.pt_share.median()) if len(sub) else np.nan,
                           'seg_class tubule_proximal': int(sub.seg_class.eq('tubule_proximal').sum()),
                           'seg_class other': int((~sub.seg_class.eq('tubule_proximal')).sum())})
PLACEMENT = pd.DataFrame(PLACE_ROWS)
PLACEMENT.to_csv(tables / 'placement_c19_c15.csv', index=False)
pd.concat({k: v for k, v in PLACED.items()}).to_csv(tables / 'placement_units.csv')
c19_package_raw = POSITIONS.loc[C19, 'pseudospace']
print(f"c19 at the package positions: {int((c19_package_raw <= CUT).sum())} of 80 at raw ≤ c = {CUT:.3f}; "
      f"median {c19_package_raw.median():.3f}")
display(PLACEMENT.groupby('cluster')[['units', 'on the PT stretch', 'on stretch, placed PT-S1', 'on stretch, placed PT-S2',
                                      'on stretch, placed PT-S3', 'seg_class tubule_proximal']].sum())
display(PLACEMENT)
print('c19 onto PT (arm ii): placed labels', onto_pt.majority.value_counts().to_dict(),
      f'· median placed raw position {onto_pt.median_value.median():.3f}')

## 8 · (c) PT unit density along the axis

- **Units:** all 15,013 export PT units, untrimmed; raw positions mapped by the primary support min–max and then to
  segment units.
- **Bins:** 10 bins cut at the equal-specimen deciles of the control units' raw positions; the **late share** (last two
  bins) is A1's statistic. Tail: beyond the control equal-specimen 99th percentile.
- **Context:** PT units per section and per 1,000 units on the curve; segmentation `tubule_proximal` polygons with
  bins and the share stage 03 labels PT; deep-S3 count (notebook 45's rule: mean distance to the 3 nearest glomerulus
  polygons of the section; deep when beyond the section's S1 95th percentile); in control S3 units the Spearman of
  position against depth (x/y used for validation only).
- **Accounting:** the IR late-PT deficit against the IR c19 + c15 units placed on the late PT stretch (section 7).

In [ ]:
from scipy.spatial import cKDTree

obs_all = UNITS.obs
raw_all = obs_all.pseudospace.to_numpy(float)
lo_raw, hi_raw = P['support_raw']
scaled_all = (raw_all - lo_raw) / (hi_raw - lo_raw)
spec_all = obs_all['sample'].astype(str).to_numpy()
seg_all = obs_all.segment_class.astype(str).to_numpy()
s_all = segment_units(scaled_all, PROF['primary']['t12'], PROF['primary']['t23'])
ctrl_all = np.isin(spec_all, CONTROLS)
EDGES = equal_weight_quantiles(raw_all[ctrl_all], spec_all[ctrl_all], np.arange(1, 10) / 10)
LATE_RAW = float(EDGES[7])
TAIL_RAW = float(equal_weight_quantiles(raw_all[ctrl_all], spec_all[ctrl_all], [.99])[0])
bins_all = np.searchsorted(EDGES, raw_all, side='right')
DENSITY = pd.DataFrame({spec: np.bincount(bins_all[spec_all == spec], minlength=10) / (spec_all == spec).sum()
                        for spec in SPECIMENS}, index=[f'decile {k + 1}' for k in range(10)]).T
DENSITY['late share'] = DENSITY[['decile 9', 'decile 10']].sum(axis=1)
DENSITY['beyond control 99th pct'] = [float(np.mean(raw_all[spec_all == s] > TAIL_RAW)) for s in SPECIMENS]
DENSITY['PT units'] = [int((spec_all == s).sum()) for s in SPECIMENS]
DENSITY['late units'] = [int(((spec_all == s) & (raw_all >= LATE_RAW)).sum()) for s in SPECIMENS]

# Context: sections, segmentation and depth
context = []
glom_depth = np.full(len(obs_all), np.nan)
for spec in SPECIMENS:
    o1 = h5_obs(run_aki / '01_units' / f'{spec}_units.h5ad', ['n_spots', 'seg_class', 'x_centroid', 'y_centroid'])
    o1.index = spec + '_' + o1.index.astype(str)
    prox = o1[o1.seg_class.astype(str).eq('tubule_proximal') & o1.n_spots.gt(0)]
    prox_pt = POSITIONS.label.reindex(prox.index).isin(['PT-S1', 'PT-S2', 'PT-S3'])
    glom = o1[o1.seg_class.astype(str).eq('glomerulus')]
    mask = spec_all == spec
    dist, _ = cKDTree(glom[['x_centroid', 'y_centroid']].to_numpy(float)).query(
        obs_all.loc[mask, ['x_centroid', 'y_centroid']].to_numpy(float), k=3)
    glom_depth[mask] = dist.mean(axis=1)
    s1_limit = np.quantile(glom_depth[mask & (seg_all == 'PT-S1')], .95)
    deep = mask & (seg_all == 'PT-S3') & (glom_depth > s1_limit)
    curve_units = int((POSITIONS['sample'].eq(spec) & POSITIONS.pseudospace.notna()).sum())
    context.append({'specimen': spec, 'PT units': int(mask.sum()), 'units on the curve': curve_units,
                    'PT units per 1,000 curve units': 1000 * mask.sum() / curve_units,
                    'segmentation tubule_proximal polygons with bins': len(prox),
                    'share of them labelled PT by stage 03': float(prox_pt.mean()),
                    'glomerulus polygons': len(glom), 'PT-S3 units': int((mask & (seg_all == 'PT-S3')).sum()),
                    'deep S3 units': int(deep.sum())})
CONTEXT = pd.DataFrame(context).set_index('specimen')
anchor = {}
for spec in CONTROLS:
    m_ = (spec_all == spec) & (seg_all == 'PT-S3')
    anchor[spec] = spearmanr(scaled_all[m_], glom_depth[m_]).statistic
m_ = ctrl_all & (seg_all == 'PT-S3')
anchor['controls pooled'] = spearmanr(scaled_all[m_], glom_depth[m_]).statistic
CONTEXT['Spearman position vs depth, control S3'] = pd.Series(anchor)

# Accounting: the IR late-PT deficit against placed c19 + c15 units on the late PT stretch
ctrl_late = DENSITY.loc[CONTROLS, 'late share'].mean()
for spec in IRS:
    placed_late = sum(int((PLACED[name]['sample'].eq(spec) & PLACED[name].on_pt_stretch
                           & PLACED[name].median_value.ge(LATE_RAW)).sum()) for name in ('c19', 'c15'))
    DENSITY.loc[spec, 'late deficit (control mean share × PT units − observed)'] = ctrl_late * DENSITY.loc[spec, 'PT units'] - DENSITY.loc[spec, 'late units']
    DENSITY.loc[spec, 'c19 + c15 placed on the late PT stretch'] = placed_late
DENSITY.to_csv(tables / 'density.csv')
CONTEXT.to_csv(tables / 'density_context.csv')
print(f'Control late cut (raw) {LATE_RAW:.4f} = {segment_units([(LATE_RAW - lo_raw) / (hi_raw - lo_raw)], PROF["primary"]["t12"], PROF["primary"]["t23"])[0]:.2f} segment units')
display(DENSITY.round(3))
display(CONTEXT.round(3))

## 9 · (d) Transport loss, the cell-cycle program and decoupling

- **Positional decoupling (primary, decision O1):** Δμ = μ_A − μ_L in segment units (A = C, L = −T_PT), for the
  condition and each pair; also the R² of A(x) on L(x) through the origin across the grid.
- **Unit coupling (descriptive):** unit scores s_P = log₂(10⁴ · Σ_{g∈P} counts / library + 1); within each of the 10
  control-decile bins (≥ 20 units) and each specimen, Spearman ρ(s_T_PT, s_C); median over bins.
- **Cell-cycle readings (5.4, decision O6):** B1 phase balance G2M′ − E2F′ at the condition's C peak; B2 mitotic genes;
  B3 arrest genes; B4 unit coupling ρ(s_G2M′, s_Cdkn1a) against ρ(s_G2M′, s_E2F′) in IR units. Computed only where
  A4-activation holds; a gene list "holds" when ≥ 2/3 of its genes in U76 meet the rule (notebook 45's share).

In [ ]:
def decoupling(prof):
    summ = prof['summary']
    rows = []
    for cname in ['condition', *PAIR_NAMES, 'relabeling 1', 'relabeling 2']:
        if (cname, 'loss T_PT') not in summ.index or (cname, 'C') not in summ.index:
            continue
        mu_l, mu_a = summ.loc[(cname, 'loss T_PT'), 'centre_of_mass'], summ.loc[(cname, 'C'), 'centre_of_mass']
        a, l_ = prof['profiles'][(cname, 'C')], prof['profiles'][(cname, 'loss T_PT')]
        k = (a @ l_) / (l_ @ l_) if (l_ @ l_) > 0 else np.nan
        r2 = 1 - ((a - k * l_) ** 2).sum() / (a ** 2).sum() if np.isfinite(k) and (a ** 2).sum() > 0 else np.nan
        rows.append({'contrast': cname, 'mu loss T_PT': mu_l, 'mu C': mu_a, 'Delta mu (C - loss)': mu_a - mu_l,
                     'R2 of A on L through the origin': r2})
    return pd.DataFrame(rows).set_index('contrast')


def unit_scores(d, members):
    cols = pd.Index(d['genes']).get_indexer([g for g in members if g in set(d['genes'])])
    total = np.asarray(d['counts'][:, cols].sum(axis=1)).ravel()
    return np.log2(1e4 * total / d['library'] + 1)


def coupling(d, a, b, specimens):
    bins = np.searchsorted(EDGES, d['position_raw'], side='right')
    rows = []
    for spec in specimens:
        for k in range(10):
            m_ = (d['specimen'] == spec) & (bins == k)
            if m_.sum() >= 20:
                rows.append({'specimen': spec, 'bin': k + 1, 'units': int(m_.sum()), 'rho': spearmanr(a[m_], b[m_]).statistic})
    return pd.DataFrame(rows)


DECOUPLING = {}
if 'T_PT' in PROGRAMS:
    DECOUPLING['primary'] = decoupling(PROF['primary'])
    DECOUPLING['primary'].to_csv(tables / 'decoupling.csv')
    display(DECOUPLING['primary'].round(3))
    score_t, score_c = unit_scores(P, PROGRAMS['T_PT']), unit_scores(P, PROGRAMS['C'])
    UNIT_COUPLING = coupling(P, score_t, score_c, SPECIMENS)
    UNIT_COUPLING.to_csv(tables / 'unit_coupling.csv', index=False)
    display(UNIT_COUPLING.groupby('specimen').rho.median().to_frame('median ρ(s_T_PT, s_C) over bins').round(3))

## 10 · Cohort facts (inventory item 35; descriptive)

Pseudobulk log₂ CPM (mt-free library) per specimen over the analysed units, whole PT and per segment. Over U76 genes
with log₂CPM ≥ 1 in every specimen: the SD of IR2A2 − IR2A4 and of Ctrl1A2 − Ctrl1A4, and their Spearman (the A2/A4
factor). These describe the data; they do not test the user's statement of equal severity.

In [ ]:
FACT_GENES = [g for g in [*INJURY_MARKERS, 'Pcna', 'Mki67', 'Top2a', *DISPLAY_TRANSPORTERS] if g in P['genes']]
fact_rows, marker_rows = [], []
for part in ('whole PT', 'S1', 'S2', 'S3'):
    cpm = {}
    for spec in SPECIMENS:
        mask = (P['specimen'] == spec) & ((P['segment'] == part) if part != 'whole PT' else True)
        cpm[spec] = np.log2(np.asarray(P['counts'][mask].sum(axis=0)).ravel() / P['library'][mask].sum() * 1e6 + 1)
    cpm = pd.DataFrame(cpm, index=P['genes'])
    keep = cpm.ge(1).all(axis=1)
    ir_diff, ctrl_diff = (cpm.IR2A2 - cpm.IR2A4)[keep], (cpm.Ctrl1A2 - cpm.Ctrl1A4)[keep]
    fact_rows.append({'part': part, 'genes (log2CPM ≥ 1 everywhere)': int(keep.sum()), 'SD IR2A2 − IR2A4': ir_diff.std(),
                      'SD Ctrl1A2 − Ctrl1A4': ctrl_diff.std(), 'Spearman of the two differences (A2/A4 factor)': spearmanr(ir_diff, ctrl_diff).statistic})
    marker_rows.append(cpm.loc[FACT_GENES].assign(part=part))
COHORT_FACTS = pd.DataFrame(fact_rows).set_index('part')
COHORT_MARKERS = pd.concat(marker_rows).reset_index().rename(columns={'index': 'gene'})
COHORT_FACTS.to_csv(tables / 'cohort_facts.csv')
COHORT_MARKERS.to_csv(tables / 'cohort_facts_markers.csv', index=False)
display(COHORT_FACTS.round(3))
display(COHORT_MARKERS[COHORT_MARKERS.part.eq('whole PT')].set_index('gene').drop(columns='part').round(2))

## 11 · Sensitivities

- **c19 arm (i):** c19 added at its package positions; its segment label is read from the control cuts on the raw
  scale. **Arm (ii):** c19 placed onto the PT (section 7). Each arm reruns sections 2–6 with U76 as the candidate
  universe (detection re-applied on the new cohort). Descriptive, not replacements.
- **Package DPT:** `dpt_pseudotime` as the position (decision D3), full refit, A2–A5 statistics.

In [ ]:
import anndata as ad

t12_raw = lo_raw + PROF['primary']['t12'] * (hi_raw - lo_raw)
t23_raw = lo_raw + PROF['primary']['t23'] * (hi_raw - lo_raw)
qc_rows = qc_index.get_indexer(C19)
c19_counts = QC_COUNTS[qc_rows].astype(float)
c19_sample = PASS1_OBS.loc[C19, 'sample'].astype(str).to_numpy()


def with_c19(positions, segments):
    obs = pd.DataFrame({'sample': c19_sample, 'group': np.where(np.isin(c19_sample, CONTROLS), 'Control', 'IR'),
                        'segment_class': segments, 'pseudospace': positions}, index=C19)
    base = UNITS.obs[['sample', 'group', 'segment_class', 'pseudospace']].astype({'sample': str, 'group': str, 'segment_class': str})
    return ad.AnnData(X=sparse.vstack([UNITS.X, c19_counts]).tocsr(), obs=pd.concat([base, obs]),
                      var=pd.DataFrame(index=UNITS.var_names))


arm_i_raw = c19_package_raw.to_numpy(float)
arm_i_seg = np.where(arm_i_raw < t12_raw, 'PT-S1', np.where(arm_i_raw < t23_raw, 'PT-S2', 'PT-S3'))
COHORTS = {'primary': P}
COHORTS['c19_package'] = cohort_inputs(with_c19(arm_i_raw, arm_i_seg), U_MT, 'c19 added at its package positions (arm i)')
COHORTS['c19_placed'] = cohort_inputs(with_c19(onto_pt.median_value.to_numpy(float), onto_pt.majority.to_numpy()), U_MT,
                                      'c19 placed onto the PT by neighbours (arm ii)')
dpt_units = UNITS.copy()
dpt_units.obs['pseudospace'] = dpt_units.obs['dpt_pseudotime'].astype(float)
COHORTS['dpt'] = cohort_inputs(dpt_units, U_MT, 'package DPT as the position')
del dpt_units
t0 = time.perf_counter()
for tag in ('c19_package', 'c19_placed', 'dpt'):
    d = COHORTS[tag]
    MODELS[tag], DENOM[tag], STAT[tag] = fit_cohort(tag, d)
    TESTS[tag], CALLS[tag], row = screen(tag, d, STAT[tag])
    PASS_ROWS.append(row)
    PROF[tag] = profiles_of(d, MODELS[tag], PROGRAMS)
    PTEST[tag] = program_tests(d, STAT[tag], PROGRAMS, TESTS[tag])
    DISCRETE[tag] = discrete_view(d, PROGRAMS)
    POSCTRL[tag] = positive_control(d, MODELS[tag], PROF[tag], TESTS[tag], CALLS[tag])
    if 'T_PT' in PROGRAMS:
        DECOUPLING[tag] = decoupling(PROF[tag])
PASS_TABLE = pd.DataFrame(PASS_ROWS)
PASS_TABLE.to_csv(tables / 'pass_rule.csv', index=False)
COHORT_TABLE = pd.DataFrame({tag: {'units in the support': len(d['position']), 'IR units': int(d['case'].sum()),
                                   'c19 units kept in the support': int(pd.Index(d['obs'].index).isin(C19).sum()),
                                   'genes': len(d['genes']), 'sets': len(d['sets'])} for tag, d in COHORTS.items()}).T
COHORT_TABLE.to_csv(tables / 'cohorts.csv')
display(COHORT_TABLE)
# A1 in the c19 arms: the late share with the c19 units added at each arm's raw position (primary control cut)
A1_ARMS = {}
for tag, extra_raw in (('c19_package', arm_i_raw), ('c19_placed', onto_pt.median_value.to_numpy(float))):
    raw_x, spec_x = np.r_[raw_all, extra_raw], np.r_[spec_all, c19_sample]
    shares = pd.Series({s: float(np.mean(raw_x[spec_x == s] >= LATE_RAW)) for s in SPECIMENS})
    A1_ARMS[tag] = {'late shares': shares,
                    'holds': bool(shares[IRS].max() < shares[CONTROLS].min() and shares[IRS].mean() <= A1_FACTOR * shares[CONTROLS].mean())}
display(pd.DataFrame({tag: v['late shares'] for tag, v in A1_ARMS.items()}).round(4))
display(PASS_TABLE.round(3))
print(f'Sensitivity fits {time.perf_counter() - t0:.0f} s')

## 12 · Claims A1–A7 (protocol section 10)

"All pairs" means all 4 IR × control pairs agree in sign; "specific" means |relabeled| < 0.25·|condition| in both
relabelings. Set claims need p_emp ≤ 0.025 and the program's own relabeled one-sided p > 0.05 in both relabelings.
A claim must also hold in direction under the package DPT, otherwise it is marked "package coordinate only". A7
lists the verdicts that flip in c19 arm (ii).

In [ ]:
CRIT72 = pd.read_csv(T72 / 'criteria_aki.csv').set_index('criterion')


def bar_values(text):
    return [float(v) for v in re.findall(r'\d+\.\d+', str(text))]


p1w = bar_values(CRIT72.loc['C3', 'value'])
genefold = bar_values(CRIT72.loc['C4', 'value'])
WITHIN_BELOW = min(p1w) < .228 + .05 or min(genefold) < .70     # 72's human vs mouse bars for C3 and C4
WITHIN_NOTE = (f"notebook 72 AKI within-segment order: P1w control/IR {CRIT72.loc['C3', 'value']} (bar 0.278), gene-fold "
               f"{CRIT72.loc['C4', 'value']} (bar 0.70)")


def stat_rule(summ, program, column, *, positive=True):
    cond = summ.loc[('condition', program), column]
    pairs = [summ.loc[(p, program), column] for p in PAIR_NAMES]
    rel = [summ.loc[(r, program), column] for r in ('relabeling 1', 'relabeling 2')]
    all_pairs = all(v > 0 for v in pairs) if positive else all(np.sign(v) == np.sign(cond) for v in pairs)
    specific = all(abs(v) < RATIO * abs(cond) for v in rel)
    return cond, pairs, rel, all_pairs, specific


def claims_for(tag):
    d, prof, summ, pt = COHORTS[tag], PROF[tag], PROF[tag]['summary'], PTEST[tag]
    out = {}
    late = None
    # A3
    if 'T_PT' in PROGRAMS:
        c, pairs, rel, ap, sp = stat_rule(summ, 'loss T_PT', 'level')
        out['A3 loss'] = {'value': c, 'pairs': pairs, 'relabeled': rel, 'all pairs': ap, 'specific': sp,
                          'holds': bool(ap and c >= EFFECT and sp)}
        c, pairs, rel, ap, sp = stat_rule(summ, 'loss T_PT', 'drift')
        setok = pt.loc['loss T_PT', 'p_emp (condition)'] <= SET_CUT and bool(pt.loc['loss T_PT', 'own relabeled p > 0.05 (both)'])
        out['A3 progressive'] = {'value': c, 'pairs': pairs, 'relabeled': rel, 'all pairs': ap, 'specific': sp,
                                 'p_emp': pt.loc['loss T_PT', 'p_emp (condition)'], 'set rule': setok,
                                 'holds': bool(ap and c >= EFFECT and sp and setok)}
    # A4
    c, pairs, rel, ap, sp = stat_rule(summ, 'C', 'level')
    out['A4 activation'] = {'value': c, 'pairs': pairs, 'relabeled': rel, 'all pairs': ap, 'specific': sp,
                            'holds': bool(ap and c >= EFFECT and sp)}
    peak = summ.loc[('condition', 'C'), 'peak']
    pair_peaks = [summ.loc[(p, 'C'), 'peak'] for p in PAIR_NAMES]
    peaks_ok = all(abs(v - peak) <= SEG_TOL for v in pair_peaks)
    setok = pt.loc['C', 'p_emp (condition)'] <= SET_CUT and bool(pt.loc['C', 'own relabeled p > 0.05 (both)'])
    out['A4 location'] = {'value': peak, 'pairs': pair_peaks, 'peaks agree': peaks_ok, 'p_emp': pt.loc['C', 'p_emp (condition)'],
                          'set rule': setok, 'holds': bool(peaks_ok and setok)}
    # A5
    if 'T_PT' in PROGRAMS:
        dec = DECOUPLING[tag]
        dmu = dec.loc['condition', 'Delta mu (C - loss)']
        pair_dmu = [dec.loc[p, 'Delta mu (C - loss)'] for p in PAIR_NAMES]
        same = all(np.sign(v) == np.sign(dmu) for v in pair_dmu) and np.isfinite(dmu)
        out['A5 decoupling'] = {'value': dmu, 'pairs': pair_dmu, 'all pairs': same,
                                'holds': bool(out['A3 loss']['holds'] and out['A4 activation']['holds'] and same and abs(dmu) >= EFFECT)}
    # Within-segment
    for prog in (['loss T_PT'] if 'T_PT' in PROGRAMS else []) + ['C']:
        for seg in ('S1', 'S2', 'S3'):
            c, pairs, rel, ap, sp = stat_rule(summ, prog, f'within_{seg}', positive=False)
            out[f'within {seg} · {prog}'] = {'value': c, 'pairs': pairs, 'relabeled': rel, 'all pairs': ap, 'specific': sp,
                                             'holds': bool(ap and abs(c) >= EFFECT and sp)}
    # A2
    out['A2 positive control'] = {'value': POSCTRL[tag][0]['share in S3'], 'holds': bool(POSCTRL[tag][2])}
    return out


CLAIMS = {tag: claims_for(tag) for tag in COHORTS}

# A1 (primary units only; c19 arms reported in the sensitivity table)
late_ctrl = DENSITY.loc[CONTROLS, 'late share']
late_ir = DENSITY.loc[IRS, 'late share']
a1_holds = bool(late_ir.max() < late_ctrl.min() and late_ir.mean() <= A1_FACTOR * late_ctrl.mean())
covered = bool((DENSITY.loc[IRS, 'c19 + c15 placed on the late PT stretch']
                >= DENSITY.loc[IRS, 'late deficit (control mean share × PT units − observed)']).all())


def direction_holds(name, primary, dpt):
    if name == 'A4 location':
        return abs(dpt['value'] - primary['value']) <= SEG_TOL
    if name == 'A2 positive control':
        return dpt['holds']
    return np.sign(dpt['value']) == np.sign(primary['value'])


claim_rows = [{'claim': 'A1 fewer PT-labelled units in the late PT', 'statistic': 'late share (last two control deciles)',
               'condition value': f"IR {', '.join(f'{v:.3f}' for v in late_ir)} vs control {', '.join(f'{v:.3f}' for v in late_ctrl)}",
               'verdict': ('claimable' if a1_holds else 'not claimable') + (' · consistent with relabelling (placed c19 + c15 cover the deficit)' if a1_holds and covered else ''),
               'DPT direction': 'n/a (counts of units, not of positions on one curve)',
               'c19 arm (ii)': ('flips' if A1_ARMS['c19_placed']['holds'] != a1_holds else 'same verdict')
                               + ' (IR ' + ', '.join(f'{v:.3f}' for v in A1_ARMS['c19_placed']['late shares'][IRS]) + ')'}]
for name, primary in CLAIMS['primary'].items():
    dpt = CLAIMS['dpt'].get(name)
    arm = CLAIMS['c19_placed'].get(name)
    dpt_ok = direction_holds(name, primary, dpt) if dpt is not None else False
    verdict = 'claimable' if primary['holds'] else 'not claimable'
    if primary['holds'] and not dpt_ok:
        verdict += ' · package coordinate only'
    if name.startswith('within') and primary['holds'] and WITHIN_BELOW:
        verdict += " · worded 'along the axis within the label'"
    claim_rows.append({'claim': name, 'statistic': {'A3 loss': 'level of L (log2)', 'A3 progressive': 'drift of L (log2)',
                                                    'A4 activation': 'level of A (log2)', 'A4 location': 'peak of A (segment units)',
                                                    'A5 decoupling': 'Delta mu = mu_A - mu_L (segment units)',
                                                    'A2 positive control': 'share of markers rising most in S3'}.get(name, 'within-segment change (log2)'),
                       'condition value': f"{primary['value']:.3f}",
                       'pairs': ', '.join(f'{v:.3f}' for v in primary.get('pairs', [])),
                       'relabeled': ', '.join(f'{v:.3f}' for v in primary.get('relabeled', [])),
                       'all pairs': primary.get('all pairs', primary.get('peaks agree')), 'specific': primary.get('specific'),
                       'p_emp': primary.get('p_emp'), 'set rule': primary.get('set rule'), 'verdict': verdict,
                       'DPT direction': ('holds' if dpt_ok else 'does not hold') + (f" ({dpt['value']:.3f})" if dpt is not None else ''),
                       'c19 arm (ii)': ('flips' if arm is not None and arm['holds'] != primary['holds'] else 'same verdict')
                                       + (f" ({arm['value']:.3f})" if arm is not None else '')})
CLAIMS_TABLE = pd.DataFrame(claim_rows)
CLAIMS_TABLE.to_csv(tables / 'claims.csv', index=False)
display(CLAIMS_TABLE)
print(WITHIN_NOTE, '· within-segment wording restricted:', WITHIN_BELOW)

# Sensitivity summary (c19 arms, DPT)
SENS_ROWS = []
for tag in ('primary', 'c19_package', 'c19_placed', 'dpt'):
    row = {'cohort': tag}
    for name, v in CLAIMS[tag].items():
        if not name.startswith('within'):
            row[f'{name} value'] = v['value']
            row[f'{name} holds'] = v['holds']
    SENS_ROWS.append(row)
SENSITIVITY = pd.DataFrame(SENS_ROWS).set_index('cohort')
SENSITIVITY.to_csv(tables / 'sensitivity.csv')
display(SENSITIVITY.T)

### 12.1 · The two readings of the cell-cycle program (5.4)

Computed only where A4-activation holds. Verdict: division-compatible if B1 is balanced, B2 holds and B3 fails;
arrest-compatible if B1 is G2/M-dominant and B3 holds; otherwise undetermined. Stated as compatibility, not mechanism.

In [ ]:
READINGS = None
if CLAIMS['primary']['A4 activation']['holds']:
    pr = PROF['primary']
    peak_i = int(np.argmax(pr['profiles'][('condition', 'C')]))
    b1 = {c: pr['profiles'][(c, "G2M'")][peak_i] - pr['profiles'][(c, "E2F'")][peak_i] for c in ['condition', *PAIR_NAMES]}
    balanced = abs(b1['condition']) < EFFECT
    g2m_dominant = all(b1[p] >= EFFECT for p in PAIR_NAMES)
    s_ = pr['s']
    e2f_peak = pr['summary'].loc[('condition', "E2F'"), 'peak']
    g2m_peak = pr['summary'].loc[('condition', "G2M'"), 'peak']

    def gene_rule(genes, target_peak):
        rows = []
        for g in genes:
            if g not in P['genes']:
                continue
            gi = P['genes'].index(g)
            cond_c, cond_ok, _ = pr['contrasts']['condition']
            levels = [pr['contrasts'][p][0][gi].mean() / np.log(2) for p in PAIR_NAMES]
            gpeak = s_[int(np.argmax(cond_c[gi]))]
            ok = all(v >= EFFECT for v in levels) and abs(gpeak - target_peak) <= SEG_TOL
            rows.append({'gene': g, 'condition level (log2)': cond_c[gi].mean() / np.log(2), 'min pair level': min(levels),
                         'peak (segment units)': gpeak, 'meets rule': ok})
        return pd.DataFrame(rows).set_index('gene')

    b2, b3 = gene_rule(MITOTIC, e2f_peak), gene_rule(ARREST, g2m_peak)
    b2_holds, b3_holds = b2['meets rule'].mean() >= GENE_SHARE, b3['meets rule'].mean() >= GENE_SHARE
    ir_mask = P['case']
    s_g2m, s_e2f = unit_scores(P, PROGRAMS["G2M'"]), unit_scores(P, PROGRAMS["E2F'"])
    s_p21 = unit_scores(P, ['Cdkn1a'])
    b4 = pd.DataFrame({'rho(G2M\', Cdkn1a)': coupling(P, s_g2m, s_p21, IRS).groupby('specimen').rho.median(),
                       'rho(G2M\', E2F\')': coupling(P, s_g2m, s_e2f, IRS).groupby('specimen').rho.median()})
    verdict = ('division-compatible' if balanced and b2_holds and not b3_holds else
               'arrest-compatible' if g2m_dominant and b3_holds else 'undetermined')
    READINGS = pd.Series({'B1 G2M\' − E2F\' at the C peak (condition)': b1['condition'],
                          'B1 pairs': ', '.join(f'{b1[p]:.3f}' for p in PAIR_NAMES),
                          'B1 balanced (|condition| < 0.25)': balanced, 'B1 G2/M-dominant (all pairs ≥ 0.25)': g2m_dominant,
                          'B2 mitotic genes meeting the rule': f"{int(b2['meets rule'].sum())} of {len(b2)}", 'B2 holds': b2_holds,
                          'B3 arrest genes meeting the rule': f"{int(b3['meets rule'].sum())} of {len(b3)}", 'B3 holds': b3_holds,
                          'verdict': verdict}, name='value')
    READINGS.to_frame().to_csv(tables / 'cell_cycle_readings.csv')
    pd.concat({'B2 mitotic': b2, 'B3 arrest': b3}).to_csv(tables / 'cell_cycle_reading_genes.csv')
    b4.to_csv(tables / 'cell_cycle_reading_b4.csv')
    display(READINGS.to_frame())
    display(pd.concat({'B2 mitotic': b2, 'B3 arrest': b3}).round(3))
    display(b4.round(3))
else:
    print('A4-activation does not hold: the cell-cycle readings are not computed (protocol 5.4).')

## 13 · Figure 6

(a) PT unit density along the axis per specimen (segment units; control blue, IR magenta); the bars give the late
share. (b) T_PT members: centred log₂ IR/control along the axis (rows by position of maximal loss), with the loss
profile L(x) (condition thick, pairs thin) and the discrete view. (c) The cell-cycle program A(x) with G2M′ and E2F′,
and the injury markers R. (d) L and A scaled to their maxima, with centres of mass; Δμ per pair. (e) Specificity:
the screen's pass row (it fails calibration) and, per claim statistic, the condition value against both relabelings.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import gaussian_kde

DIVERGING = LinearSegmentedColormap.from_list('div', ['#3a5fa8', '#e9e8e4', '#b6492a'])
pr = PROF['primary']
fig = plt.figure(figsize=(180 * MM, 132 * MM))
gs = fig.add_gridspec(2, 3, hspace=.5, wspace=.45)

# (a) density
ax = fig.add_subplot(gs[0, 0])
xs = np.linspace(-.3, 3.6, 400)
ctrl_s = s_all[ctrl_all]
bw = gaussian_kde(ctrl_s).factor * ctrl_s.std(ddof=1)
late_s = float(segment_units([(LATE_RAW - lo_raw) / (hi_raw - lo_raw)], PROF['primary']['t12'], PROF['primary']['t23'])[0])
ax.axvspan(late_s, xs[-1], color=LIGHT, alpha=.45, lw=0, zorder=0)
for spec in SPECIMENS:
    v = s_all[spec_all == spec]
    kde = gaussian_kde(v, bw_method=bw / v.std(ddof=1))
    color = CONTROL_COLOR if spec in CONTROLS else IR_COLOR
    ax.plot(xs, kde(xs), color=color, lw=1, ls='-' if spec.endswith('A2') else '--', label=spec)
for b in (1, 2):
    ax.axvline(b, color=MUTED, lw=.4, ls=':', zorder=0)
ax.set_ylim(0, .9)
ax.set(xlabel='segment units (S1 0–1, S2 1–2, S3 2–3)', ylabel='PT unit density', title='a · PT units along the axis')
ax.legend(loc='upper center', ncol=4, handlelength=1.4, columnspacing=.8, fontsize=5.5)
ax.text(.02, .86, 'late share (shaded): control ' + '/'.join(f"{DENSITY.loc[s, 'late share']:.3f}" for s in CONTROLS)
        + ', IR ' + '/'.join(f"{DENSITY.loc[s, 'late share']:.3f}" for s in IRS)
        + '\ndeep S3 units: control ' + '/'.join(str(int(CONTEXT.loc[s, 'deep S3 units'])) for s in CONTROLS)
        + ', IR ' + '/'.join(str(int(CONTEXT.loc[s, 'deep S3 units'])) for s in IRS),
        transform=ax.transAxes, va='top', fontsize=5, color=MUTED)

# (b) heatmap of T_PT and the loss profile
if 'T_PT' in PROGRAMS:
    sub = gs[0, 1].subgridspec(2, 1, height_ratios=[1, 1.6], hspace=.08)
    axp, axh = fig.add_subplot(sub[0]), fig.add_subplot(sub[1])
    for p in PAIR_NAMES:
        axp.plot(pr['s'], pr['profiles'][(p, 'loss T_PT')], color=IR_COLOR, lw=.5, alpha=.5)
    axp.plot(pr['s'], pr['profiles'][('condition', 'loss T_PT')], color=IR_COLOR, lw=1.4)
    for k, part in enumerate(('S1', 'S2', 'S3')):
        axp.hlines(DISCRETE['primary'].loc['loss T_PT', part], k, k + 1, color=INK, lw=.8, ls=':')
    axp.axhline(0, color=LIGHT, lw=.6)
    axp.set_xlim(pr['s'][0], pr['s'][-1])
    axp.set(ylabel='L(x), log₂', title='b · Transport loss (T_PT, 93 genes)')
    axp.tick_params(labelbottom=False)
    c_, ok_, _ = pr['contrasts']['condition']
    rows = [P['genes'].index(g) for g in PROGRAMS['T_PT'] if ok_[P['genes'].index(g)]]
    mat = c_[rows] / np.log(2)
    order = np.argsort(np.argmin(mat, axis=1))
    im = axh.imshow(np.clip(mat[order], -2, 2), aspect='auto', cmap=DIVERGING, vmin=-2, vmax=2,
                    extent=[pr['s'][0], pr['s'][-1], len(rows), 0], interpolation='nearest')
    axh.set(xlabel='segment units', ylabel=f'{len(rows)} transporters')
    axh.set_yticks([])
    for b in (1, 2):
        axh.axvline(b, color=INK, lw=.4, ls=':')
        axp.axvline(b, color=MUTED, lw=.4, ls=':', zorder=0)
    cax = axh.inset_axes([1.02, .1, .04, .8])
    cb = fig.colorbar(im, cax=cax)
    cb.set_label('log₂', fontsize=5)
    cb.ax.tick_params(labelsize=5)

# (c) cell-cycle program (top) and injury markers (bottom)
sub = gs[0, 2].subgridspec(2, 1, height_ratios=[1.6, 1], hspace=.12)
ax, axr = fig.add_subplot(sub[0]), fig.add_subplot(sub[1])
for p in PAIR_NAMES:
    ax.plot(pr['s'], pr['profiles'][(p, 'C')], color=IR_COLOR, lw=.5, alpha=.5)
    axr.plot(pr['s'], pr['profiles'][(p, 'R')], color=IR_COLOR, lw=.5, alpha=.5)
ax.plot(pr['s'], pr['profiles'][('condition', 'C')], color=IR_COLOR, lw=1.4, label='cell-cycle program C')
ax.plot(pr['s'], pr['profiles'][('condition', "G2M'")], color=INK, lw=.8, ls='--', label='G2-M only')
ax.plot(pr['s'], pr['profiles'][('condition', "E2F'")], color=MUTED, lw=.8, ls=':', label='E2F only')
axr.plot(pr['s'], pr['profiles'][('condition', 'R')], color=IR_COLOR, lw=1.4)
for a_ in (ax, axr):
    a_.axhline(0, color=LIGHT, lw=.6)
    a_.set_xlim(pr['s'][0], pr['s'][-1])
    for b in (1, 2):
        a_.axvline(b, color=MUTED, lw=.4, ls=':', zorder=0)
ax.tick_params(labelbottom=False)
ax.set(ylabel='centred log₂', title='c · Cell-cycle program and injury markers')
ax.legend(loc='upper left', fontsize=5.5)
axr.set(xlabel='segment units', ylabel='R, log₂')

# (d) decoupling
ax = fig.add_subplot(gs[1, 0])
if 'T_PT' in PROGRAMS:
    l_ = pr['profiles'][('condition', 'loss T_PT')]
    a_ = pr['profiles'][('condition', 'C')]
    ax.plot(pr['s'], l_ / np.abs(l_).max(), color=INK, lw=1.2, label='transport loss L')
    ax.plot(pr['s'], a_ / np.abs(a_).max(), color=IR_COLOR, lw=1.2, label='cell-cycle program A')
    dec = DECOUPLING['primary']
    for y, col, key in ((1.1, INK, 'mu loss T_PT'), (1.1, IR_COLOR, 'mu C')):
        ax.plot([dec.loc['condition', key]], [y], marker='v', color=col, ms=4, ls='none')
    ax.axhline(0, color=LIGHT, lw=.6)
    for b in (1, 2):
        ax.axvline(b, color=MUTED, lw=.4, ls=':', zorder=0)
    ax.set_ylim(-.05, 1.2)
    ax.set(xlabel='segment units', ylabel='profile ÷ its maximum',
           title=f"d · Δμ = {dec.loc['condition', 'Delta mu (C - loss)']:+.2f} segment; R² {dec.loc['condition', 'R2 of A on L through the origin']:.2f}")
    ax.legend(loc=(.02, .55), fontsize=5.5)
    txt = 'Δμ per pair\n' + '\n'.join(f"{p}: {dec.loc[p, 'Delta mu (C - loss)']:+.2f}" for p in PAIR_NAMES)
    ax.text(.98, .03, txt, transform=ax.transAxes, ha='right', va='bottom', fontsize=5, color=MUTED)

# (e) specificity
ax = fig.add_subplot(gs[1, 1:])
stats_e = [('loss T_PT', 'level', 'L level'), ('loss T_PT', 'drift', 'L drift'), ('C', 'level', 'A level'),
           ('C', 'drift', 'A drift'), ('R', 'level', 'R level')]
stats_e = [s for s in stats_e if ('condition', s[0]) in pr['summary'].index]
xpos = np.arange(len(stats_e))
for j, (prog, col, lab) in enumerate(stats_e):
    vals = [pr['summary'].loc[(c, prog), col] for c in ('condition', 'relabeling 1', 'relabeling 2')]
    ax.bar(j - .25, vals[0], width=.25, color=IR_COLOR, label='condition' if j == 0 else None)
    ax.bar(j, vals[1], width=.25, color=LIGHT, edgecolor=MUTED, lw=.4, label='relabeling 1' if j == 0 else None)
    ax.bar(j + .25, vals[2], width=.25, color=LIGHT, edgecolor=MUTED, lw=.4, hatch='///', label='relabeling 2' if j == 0 else None)
ax.axhline(0, color=INK, lw=.5)
ax.set_xticks(xpos, [s[2] for s in stats_e])
ax.set_ylim(top=max(4.6, ax.get_ylim()[1]))
ax.set(ylabel='log₂', title='e · Specificity: condition against both relabelings')
ax.legend(loc=(.3, .5), ncol=3)
pr_row = PASS_TABLE[PASS_TABLE.cohort.eq('primary')].iloc[0]
lines = [f"Screen (Figure 6 universe): {pr_row['calls']} calls; relabeled {pr_row['relabeled calls']}; NCDR {pr_row['NCDR']:.2f}; "
         f"calibration share {pr_row['calibration share']:.3f} → {pr_row['calibration'].upper()}"]
if 'loss T_PT' in PTEST['primary'].index:
    lines.append(f"set p_emp: T_PT {PTEST['primary'].loc['loss T_PT', 'p_emp (condition)']:.2g}, C "
                 f"{PTEST['primary'].loc['C', 'p_emp (condition)']:.2g}; both also positional under the relabelings (p < 0.05)")
ax.text(.01, .97, '\n'.join(lines), transform=ax.transAxes, ha='left', va='top', fontsize=5.5, color=INK)
save(fig, 'figure6_aki_along_pt')
plt.show()

## 14 · How the protocol was read

Where the protocol left a choice, the reading closest to notebooks 45 and 71 was taken. Nothing was tuned on results.
- **Universe.** Two `aki_inputs` calls, as specified. All 13 mt genes leave the universe here: mt-Atp8 passes AKI
  detection on the panel pool, whereas notebook 71 removed 12.
- **G5 (reuse).** Notebook 71's inputs were rebuilt by its own call (`hvm_inputs`, then `aki_inputs` on its pool) and
  matched to its cached counts digest. T_spatial, LR_spec and Δ agree at rtol 1e-6 on all 10,164 genes converged in
  both runs. α differs for 2 genes (Ints12, Cetn2) at a relative 1.4 × 10⁻⁵. α is an optimiser output, so this is
  recorded rather than a stop; the run stops only beyond 1e-4. Downstream statistics of those genes agree.
- **T.** HGNC rows with locus type "gene with protein product", unique approved symbols, mapped by upper-case human
  symbol through the ortholog map.
- **T_PT enrichment.** For every pass-2 unit, the library is the total over the one-to-one ortholog panel without
  the mt genes. The analysis library differs only by genes with no count in any analysed PT unit. "Non-PT" is every
  other pass-2 label of that control mouse, which includes its c19 units (5 in total).
- **Profiles.** A member is used when it converged and is not separated in that contrast; for a pair, also in both
  specimen fits. The within-segment change is the profile at the last grid point inside [k, k + 1] minus at the
  first.
- **p_emp.** The null is the one-sided cameraPR p of every U76 set under both relabelings (2 × 1,495).
- **Positive control.** The Mann–Whitney test uses one-sided cameraPR p. The continuous argmax uses the uncentred
  Δ_g(x), as 5.2 defines Δ.
- **Placement.** For c19, the reference excludes c19 itself (its units would otherwise be each other's neighbours).
  For c15, the reference includes c19.
- **c19 arms.**
  - Arm (i) labels the c19 units by the control cuts on the raw scale; arm (ii) by the majority of their 30 nearest
    PT units.
  - Both arms take U76 with mt as candidates and re-apply detection (arm ii drops 1 gene).
  - A1 in the arms adds the c19 positions to all PT units and keeps the primary control cut.
- **DPT direction.** The sign of level, drift, Δμ and within-segment change; a peak within 0.5 segment, each on its
  own segment scale; for A2, detection.
- **Within-segment wording.** Uses notebook 72's AKI C3 (P1w) and C4 (gene-fold) against the human vs mouse bars.
- **5.4.**
  - A gene's Δ is its centred level over the grid.
  - B2 and B3 hold when ≥ 2/3 of the listed genes in U76 meet the rule (notebook 45's share).
  - B1 "balanced" is judged on the condition value.
- **Density.** The KDE uses Scott's factor on the pooled control segment units, at the same absolute bandwidth for
  every specimen (display only). A1's "covered" flag compares each IR mouse's deficit with its own c19 + c15 units
  placed on the late PT stretch.
- **Claims table.** The "DPT direction" and "c19 arm (ii)" columns are informational for claims that are not
  claimable.

## 15 · Summary

**Inputs.**
- 15,013 package PT units, 14,048 inside the common support.
- U76 holds 10,701 genes: notebook 71's 10,198 plus 503. It admits all 18 of notebook 45's injury markers (71: 14),
  Mki67, Top2a and Pcna. 468 genes without a one-to-one ortholog pass detection but are not tested.
- 1,495 pathways.
- Every guard passed. Notebook 71's fits are reproduced (α of 2 genes within 1.4 × 10⁻⁵ relative; section 14).

**The screen** (Figure 6 universe; notebook 71's calls stay the method-comparison record).
- 106 calls, with 20 and 30 relabeled; NCDR 0.236 (passes the relabeling rule).
- Calibration share 0.144: **fails** (notebook 71: 0.120).
- The gene-level calibration share is 0.062, within 0.075.
- Against notebook 71: z Spearman 0.999; 100 calls kept, 3 lost, 5 gained.

**Claims (protocol section 10).**

| Claim | Statistic (condition; 4 pairs; 2 relabelings) | Verdict |
|---|---|---|
| A1 fewer PT-labelled units late | late share: IR 0.200, 0.138 vs control 0.204, 0.196 | not claimable (IR2A2 is not below the controls) |
| A2 positive control | 17 of 18 markers rise most in S3; 6 of 13 injury Hallmarks called; Mann–Whitney p 1.3 × 10⁻⁵ | detection passes; specificity fails notebook 45's rule, passes R4 |
| A3 transport loss | L̄ 0.642; 0.71, 0.86, 0.43, 0.56; 0.174, 0.032 | not claimable: relabeling 1 (A2/A4) is 0.27 of the condition |
| A3 progressive | d_L 0.621; 0.92, 0.90, 0.45, 0.46; 0.124, 0.086; T_PT p_emp 0.0003 | not claimable: T_PT is also positional under both relabelings (p ≤ 0.0001) |
| A4 cell-cycle program | Ā 0.798; 0.78, 0.76, 0.76, 0.74; 0.017, 0.035 | **claimable** (DPT 0.943; c19 arms 0.795–0.799) |
| A4 location | peak 2.75 segment units; pairs 2.36, 2.36, 2.80, 2.80; C p_emp 0.014 | not claimable: the position of activation differs between the IR mice (relabeled p < 0.0001) |
| A5 decoupling | Δμ +0.07; pairs −0.04, +0.03, +0.07, +0.14 | not claimable: no positional decoupling resolved. A(x) is nearly proportional to L(x) (R² 0.97) |
| Within-segment | e.g. loss within S3 0.86 (relabeled 0.27); C within S2 0.56 (0.16, 0.15) | none claimable (specificity) |

**Descriptive findings.**
- **Deep S3 is depleted.** IR sections keep 251 and 32 deep-S3 units, against 783 and 700 in the controls. In
  control S3, position correlates with depth (Spearman 0.43, 0.62).
- **The late share is not depleted.** IR units leave the ninth control decile (0.019, 0.030 against 0.08), but
  IR2A2 piles up in the tenth (0.181).
- **Unit coupling.** The median ρ(s_T_PT, s_C) is −0.32 and −0.17 in IR, against −0.16 and −0.04 in the controls.
- **Cohort facts.** Whole-PT SD of the IR pair difference is 0.175, against 0.249 for the controls. The A2/A4 factor
  is ρ 0.40. In S3 the IR pair diverges: SD 0.49 against 0.29.

**Cell-cycle reading (5.4): undetermined.** The program is E2F-dominant at its peak (G2M′ − E2F′ = −0.75), so it is
neither balanced nor G2/M-dominant. Mitotic genes meet their rule (6 of 7); arrest genes do not (2 of 4).

**c19 and c15.**
- c19: 21 of 80 units sit on the PT stretch.
- c15: 569 of 2,425 units sit on the PT stretch, 556 of them placed in S3; 501 are in the IR mice.
- Adding c19 (either arm) changes no verdict.
- The IR2A4 late deficit (342 units) exceeds its placed c19 + c15 units (250).

**What this cannot tell you.**
- With 2 + 2 mice everything is descriptive.
- The AKI screen fails calibration.
- Transport loss and the position of the cell-cycle program are not specific beyond the between-mouse differences
  (relabeling 1 carries the A2/A4 factor).
- The IRI time point is unknown, so the cell-cycle program is not interpreted as proliferation or arrest.

## 16 · Run record

In [ ]:
import platform

from rpy2.robjects import r as R

cache_entries = []
for sidecar in sorted(cache.glob('*.key.json'), key=lambda p: json.loads(p.read_text())['written_utc']):
    meta = json.loads(sidecar.read_text())
    if meta['stage'] in SECONDS:
        cache_entries.append({'stage': meta['stage'], 'key': meta['key'], 'written_utc': meta['written_utc'],
                              'compute seconds': round(SECONDS[meta['stage']], 1)})
CACHE_TABLE = pd.DataFrame(cache_entries).drop_duplicates('stage', keep='last').sort_values('stage').reset_index(drop=True)
display(CACHE_TABLE)
record = {
    'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'protocol': 'docs/results/pt-package-aki.md (commit 21dbc94)',
    'inputs': PROVENANCE, 'universe': UNIVERSE_TABLE.astype(str).to_dict(), 'n_jobs': N_JOBS, 'mt_genes': MT,
    'guards': guard_log, 'hgnc_present': HGNC_OK, 'pending': T_PENDING,
    'programs': {k: len(v) for k, v in PROGRAMS.items()}, 'pass_rule': PASS_TABLE.to_dict(orient='records'),
    'compare71': COMPARE71.astype(str).to_dict(), 'claims': CLAIMS_TABLE.astype(str).to_dict(orient='records'),
    'cell_cycle_reading': None if READINGS is None else READINGS.astype(str).to_dict(),
    'segment_cuts': {tag: [PROF[tag]['t12'], PROF[tag]['t23']] for tag in PROF},
    'versions': {'python': platform.python_version(), 'R': str(R('R.version.string')[0]),
                 'limma': str(R('as.character(packageVersion("limma"))')[0]),
                 **{m: version(m) for m in ('numpy', 'scipy', 'pandas', 'anndata', 'rpy2', 'matplotlib', 'pseudospace-reconstruction')}},
    'gene_model': {'module': 'pseudospace.pathway_pipelines.standard_gene_model', 'basis_df': BASIS_DF,
                   'grid_points': GRID_POINTS, 'nb_gam': nb_gam.GENE_MODEL},
    'stage_cache': CACHE_TABLE.to_dict(orient='records'), 'compute_seconds_total': float(sum(SECONDS.values())),
    'wall_seconds': time.perf_counter() - NOTEBOOK_START}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=str))
print(f"Wall time {record['wall_seconds']:.0f} s; summed uncached compute time of the cached stages "
      f"{record['compute_seconds_total']:.0f} s")